# Chess — ThompsonZero HYBRID (state-value head + policy-prior ordering)

Merges the two sibling chess notebooks to keep the strengths of each and drop
the weakness of each:

| | `thompson_z` (per-action) | `thompson_value` (state-value) | **this notebook (hybrid)** |
|---|---|---|---|
| value signal | per-action head, predicts values for legal-but-unseen (even *illegal*) slots | **grounded**: only ever evaluated on a visited position | **grounded** (state-value head) |
| move ordering | per-action head | none — expands *all* children (breadth) | **policy-prior head** (Bayesian ordering) |
| search shape | deep, narrow | **shallow** — one node costs ~35 child evals | **deep, narrow** — one node = one eval |
| net size | 12.7M (two 4674-wide heads) | 0.6M | 2.9M (value tiny; flat policy head is the bulk) |

### The two heads

- **VALUE head** (from `thompson_value`): one Dirichlet over the *current*
  position's (win, draw, loss) + a confidence. A value belief is *never*
  fabricated for a legal-but-unevaluated child (the failure the state-value
  design was built to fix) — every value in the tree is grounded in a real
  evaluation of a real, visited position.
- **POLICY-PRIOR head** (the AlphaZero half): logits over the legal moves + a
  scalar concentration β — a Dirichlet-Categorical belief over *which move is
  best*. It is an **ordering signal only**, never a value claim. Flat (gathered
  to legal at every boundary, so its 4674-wide output never crosses a bus) for
  robustness; a spatial from-square head is a documented, bug-prone follow-up.

### The tree: depth restored, values grounded, Thompson throughout

- **Expanding a node is ONE forward pass** on the node's own state (value +
  policy). `thompson_value` paid ~35 child evaluations to open a single node,
  which capped search depth; here one simulation opens one node, so a budget of
  N sims again buys a tree of ~N expanded nodes — deep enough for real forcing
  lines.
- **PUCT-style Thompson selection** at each node. Thompson-sample *every*
  edge's value belief (opened edges use their grounded belief; unopened edges
  use the node's own value belief V(s) as an FPU prior, pessimized by a small
  `_FPU_REDUCTION`; proven edges are exact spikes), then add a **persistent
  policy-prior bonus** `U = c_puct·P(a)·√(ΣN)/(1+N(a))` and take the argmax.
  The Thompson draw carries the *value* uncertainty (no exploration constant on
  it); `U` carries *policy-guided* exploration and decays as an edge accrues
  visits — so a high-prior move keeps getting tried until its own value
  evidence takes over. This is AlphaZero's exploration term with a Bayesian
  value posterior in place of a scalar Q.

  > **Why not pure Thompson (the original design)?** The first version let the
  > policy pick only the *first* unopened edge to open, so once all ~35 moves
  > were opened (early, at 100–400 sims) the policy had zero further influence
  > and selection was governed entirely by the untrained, near-uniform value
  > beliefs → uniform visits → a noise policy target → nothing learned. The
  > persistent `U` term (one interpretable knob, `_C_PUCT`) is what makes the
  > policy actually steer a deep search, which is the whole premise of the
  > hybrid.
- The node's propagated **value** is the P(best)-weighted mixture over its
  *opened* edges only (unopened edges never inflate it — that would reintroduce
  the maximization bias mixture propagation exists to avoid), falling back to
  V(s) when nothing is opened yet.
- **MCTS-Solver, innovation calibration (λ), tree reuse, restart curriculum,
  opponent pool** — all carried over unchanged. Terminal children are detected
  the moment an edge is opened, so proofs stay cheap; calibration's ground
  truth flows through those proofs (as in `thompson_value`).

### Training: two targets per position, prioritized replay

Each played position emits BOTH:
- a **value target** — the node's posterior (win, draw, loss) counts (the
  `thompson_value` state target), z-mixed toward the game outcome (γ-ramped),
  or an exact spike if solver-proven;
- a **policy target** — the MCTS **visit distribution** over the legal moves.

Losses are the SAME Dirichlet-Multinomial evidence NLL for both heads: the
value counts scored under Dir(conf·p), and the visit counts scored under
Dir(β·softmax(legal logits)) — both non-self-referential, so confidence and
concentration are *earned*. The policy loss runs on the gathered legal entries
(segment/scatter ops over ~35 of 4674 per row) and reuses the on-device
`index_select` + DirectML split-backward machinery.

**Prioritized replay**: every sample is tagged played-path (z-mixed, on the
actually-played trajectory) or solver-proven (exact ground truth), and
minibatches are drawn ∝ a per-class weight (`PRIORITY_PLAYED` /
`PRIORITY_SOLVER`) — true prioritized sampling, loss unweighted. Both classes
are high quality by construction (no low-value off-path interior nodes dilute
the buffer), so they default to equal weight; the knobs let you emphasize the
z-informed played path or the exact solver labels.

### Eval: one unified Elo pool over (checkpoint × sim-budget)

Every checkpoint enters a SINGLE shared Elo table as three players — `@0`
(search-free policy-head argmax), `@32`, `@128` — plus one `random` mover, so
**search-vs-no-search and generation-vs-generation are directly comparable in
one ratings table**. When a checkpoint is added, its three players each play
`EVAL_GAMES_PER_PAIR=2` games (alternating colours) against every existing
player, then `EVAL_RANDOM_MULT·N` (default 5N) random-pair games run over all
players to keep old ratings mixing. The key diagnostic — plotted directly — is
the **search benefit**: `Elo(@128) − Elo(@0)` over training. If the hybrid is
working, that gap is positive and grows; if it's flat, search isn't adding
strength and the value/policy aren't good enough yet. (`@0` alone is the
slowest-moving indicator — raw policy with no lookahead — so never read it in
isolation.) Cost scales as ~`(6 + EVAL_RANDOM_MULT)·N` games/eval and the
`@32`/`@128` search games dominate it, so `DEEP_EVAL_EVERY` is 1000 by default;
raise it (or prune old benchmarks) if evals get expensive as the pool grows.

### Recovering the solver signal (aux) at low-branching nodes

The hybrid's one-edge-at-a-time expansion almost never fully proves an interior
node, which starved both the solver-label training signal (`aux≈0`) AND the
calibration's ground-truth channel (so λ was measuring self-consistency, not
truth). Remedy: at a **low-branching node** (≤ `_FORCING_MAXLEGAL` legal moves
— in check or a sparse endgame, where terminals and proofs live), apply every
move and mark the terminal children proven — *no NN eval needed* (terminals
need none), just a few clones. This recovers `thompson_value`'s dense terminal
detection where it matters, feeds MCTS-Solver so forcing lines get proven and
emit exact solver-label samples, and feeds real ground-truth innovations to the
calibrator. `aux` should now climb off zero (especially with the restart
curriculum seeding endgames); if it stays at zero, lower `_FORCING_MAXLEGAL`'s
usefulness is limited and the value head is on its own.

### Network architecture

```
Input (20, 8, 8) → Stem (Conv-GN-ReLU) → 6 × ResBlock(64, SE) + skip
   → Head 1×1 conv (8 ch) → flatten (512) →
        VALUE : Linear(3) softmax (p_win,p_draw,p_loss) + Linear(1) softplus α₀
        POLICY: Linear(4674) logits (gathered to legal) + Linear(1) softplus β
```

### Carried over unchanged from the sibling notebooks

Value-grid Dirichlet decomposition (v = p_win − p_loss with the (1−d) draw
shrinkage), mixture propagation, MCTS-Solver (WIN > DRAW > LOSS), innovation
calibration (empirical-Bayes λ, no caps), tree reuse (`_descend`),
solver-labeled aux samples, γ-ramped Z_MIX, backward-restart curriculum +
random opponent pool, opponent-pool diversity, MP self-play + central GPU
inference server, all DirectML workarounds (`LerpFreeAdamW`, elementwise
GroupNorm/softplus, on-device gather probe, split backward, model lock, CPU
eval replicas, main-thread autograd warm-up), running-Elo tournaments,
checkpoint resume + benchmark dedup, and the self-play/buffer diagnostics.

### Failure mode fixed after the first run (why the defaults are what they are)

The first hybrid training run (~4k episodes) showed **no learning** — flat Elo,
`random` competitive with every checkpoint. Root cause was that the value head
was *starved of signal*, which starves everything downstream, compounded by
three secondary faults. The current defaults fix all four:

1. **Dense value signal (`Z_GAMMA = 1.0`, was 0.97).** The one-edge-at-a-time
   expansion almost never fully proves an interior node (`aux ≈ 0`), so —
   unlike `thompson_value`, which had a flood of solver labels — the game
   outcome `z` is the value head's *only* dense training signal. The old
   end-of-game ramp decayed `z` to ~1% on opening positions, so the value head
   had essentially nothing to learn from across most of the board. Flat `z`
   (AlphaZero-style, every position trained on the outcome) restores it. Raise
   `Z_MIX` toward 0.7 if the value head still lags.
2. **Persistent policy in selection** (the `U` term above). Without it the
   visit-count policy target was noise, so the policy head — 83% of the
   parameters — never learned and the search never concentrated.
3. **Balanced losses (`_POLICY_CONC_CAP`).** The policy visit target used to
   sum to the raw sim count (100–400), making the policy loss ~10× the value
   loss and starving the value head of gradient. Capping the target
   concentration puts the two heads on the same scale.
4. **Curbed value overconfidence (`CONF_CAP = 30`, was 100).** Predicted value
   α₀ had run away to ~95 (confidently wrong) → tight beliefs → a search that
   barely explored. The lower cap keeps beliefs wide enough for Thompson
   selection to resolve moves; watch that predicted α₀ now tracks the target
   and λ stays near 1.

If it still stalls, the diagnostic order is: is the **value** head learning
(watch `vErr` fall and the `mcts` tracks, *not* `policy(0)`, beat random)? If
not, raise `Z_MIX`. If the value head learns but the **policy** doesn't (flat
`policy(0)`, `pol_loss` not falling), raise `_C_PUCT` so visits concentrate
more. `aux` staying near zero is expected here (it is *not* a health metric in
the hybrid the way it was in `thompson_value`).

### Things to watch in a real run

- The policy head is ~83% of the parameters (a flat `Linear(512, 4674)`). If
  training is compute-bound, the spatial from-square policy head (documented in
  `thompson_z`'s intro) would shrink it dramatically — but needs its own index-
  remap verification pass before trusting it.
- Early on the policy is near-flat, so the FPU term drives broad exploration
  (many root edges open) — expected; the tree narrows as the policy sharpens.
- Transpositions are evaluated once per parent (no cross-node memo) — a
  per-search transposition table is a natural follow-up if profiling shows it
  matters.

In [ ]:
%pip install open_spiel -q
# AMD/Intel GPU acceleration via DirectML (Windows / WSL2). Needs Python
# 3.11 or 3.12 (torch-directml has no 3.13 wheel) and pins torch==2.3.1.
# Harmless if you're on CUDA/CPU — the device code below falls back.
%pip install torch-directml -q

In [ ]:
import pyspiel

# Chess is already implemented natively in OpenSpiel (C++, games/chess/) — no
# custom engine needed here, unlike the Boop notebooks. It provides exact
# detection of every real chess draw (insufficient material, threefold
# repetition, stalemate, the 50-move rule) as well as checkmate, all inside
# is_terminal()/returns(); utilities are exactly WinUtility=+1, DrawUtility=0,
# LossUtility=-1, matching this notebook's v = p_win - p_loss value scale
# with no rescaling needed.
#
# Board-plane convention (see chess.cc ObservationTensor): the 20 planes are
# ABSOLUTE, not player-relative — White/Black pieces occupy fixed planes
# regardless of who is asking, with a side-to-play plane telling the network
# whose turn it is (unlike Boop's mover-relative flat observation). This is
# also why there is no board-symmetry augmentation available for chess (a
# left-right mirror is not a symmetry of the starting position: King/Queen
# occupy distinct files) — see the intro cell for a color-flip 180° rotation
# that WOULD be a valid symmetry, left unimplemented for now.
GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = GAME.num_distinct_actions()
_OBS_SHAPE   = tuple(GAME.observation_tensor_shape())    # (20, 8, 8)

print('Game:', GAME.get_type().long_name)
print('Actions:', _NUM_ACTIONS)
print('Observation shape:', _OBS_SHAPE)
print('Utility (min/max):', GAME.min_utility(), GAME.max_utility())

_state = GAME.new_initial_state()
print()
print(_state)
print('Legal actions from the start position:', len(_state.legal_actions()))
del _state

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

# ── Device selection ──────────────────────────────────────────────────────────
# Same policy as the sibling chess notebooks: the GPU only wins with BIG
# batches, so self-play NN requests are funneled through one batching server
# and training runs large-batch steps there; small-batch paths (eval, arena)
# use CPU replicas.
DEVICE_PREFERENCE = 'directml' # 'cpu' | 'directml' | 'cuda' | 'auto'

def _pick_device(pref):
    if pref in ('directml', 'auto'):
        try:
            import torch_directml
            try:    _name = torch_directml.device_name(0)
            except Exception: _name = 'DirectML GPU'
            print(f'Using DirectML: {_name}')
            return torch_directml.device(), 'directml'
        except Exception:
            if pref == 'directml':
                print('DirectML requested but unavailable — falling back.')
    if pref in ('cuda', 'auto') and torch.cuda.is_available():
        return torch.device('cuda'), 'cuda'
    return torch.device('cpu'), 'cpu'

DEVICE, _BACKEND = _pick_device(DEVICE_PREFERENCE)


# ── Two-headed design: a STATE-VALUE head + a POLICY-PRIOR head ───────────────
# This notebook merges the two sibling designs:
#   - thompson_value: the network predicts ONE Dirichlet over the CURRENT
#     state's (win, draw, loss) outcome. A value belief is only ever claimed
#     for a position actually visited (never fabricated for a legal-but-
#     unevaluated child), and every parameter works on that one estimate.
#   - thompson_z / AlphaZero: a POLICY prior over the legal moves lets search
#     decide which few children are worth opening, so one NN call expands one
#     node and the tree can go DEEP down a narrow forcing line — instead of
#     paying ~35 child evaluations to open a single node.
#
# The value head is the outcome belief FROM THE PERSPECTIVE OF THE PLAYER TO
# MOVE; the scalar value v = p_win − p_loss ∈ [−1, 1] is chess's own utility
# scale. The policy head is a Dirichlet-CATEGORICAL belief over "which legal
# move is best": a categorical mean P(a) plus a scalar concentration β. It is
# an ORDERING signal only — never a value claim — used by the tree to Thompson-
# sample which unopened edge to open next (the Bayesian analogue of PUCT's
# policy term, with a real posterior draw in place of a tuned c_puct·√N knob).
_WIN, _DRAW, _LOSS = 0, 1, 2                       # outcome columns (win, draw, loss)
_TERM_DRAW = _DRAW                                  # alias: solver/terminal draw marker
# MCTS-Solver: a proven outcome for the mover flips one ply up (win<->loss,
# draw stays), because the parent's mover is the opponent. Indexed by outcome.
_FLIP_TERM = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)

ALPHA_FLOOR = 0.05    # per-component Dirichlet floor: keeps discretization + NLL finite
CONF_MIN    = 0.5     # smallest predictable VALUE prior strength (pseudo-observations)
CONF_MAX    = 100.0   # value confidence cap: stops runaway across generations
POL_CONF_MIN = 1.0    # smallest predictable POLICY concentration β
POL_CONF_MAX = 1000.0 # policy concentration cap (visit-count evidence tops out ~sims)


# ── Input helpers ─────────────────────────────────────────────────────────────
# Chess's observation tensor is ALREADY a dense (20, 8, 8) plane stack — no
# reshape needed. It is NOT player-relative (a side-to-play plane tells the
# network whose turn it is), which is also why there is no useful board-
# symmetry augmentation for chess — see the intro cell.

def state_to_tensor(state, device):
    """Single game state → (1, 20, 8, 8) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = obs.reshape(1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat observations → (B, 20, 8, 8) float tensor. Accepts python
    lists or numpy arrays of any float dtype (replay samples and wire traffic
    are fp16 — see the tree-ops cell) and converts in ONE pass."""
    obs = np.asarray(obs_list, dtype=np.float32)
    x   = obs.reshape(-1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


# ── Device capability probe: on-device gather (index_select) ─────────────────
# The policy head is dense (B, 4674) but only ~35 legal entries per row are
# ever wanted (both on the inference server's response path and in the policy
# training loss). Gathering those ON the device means only ~1% of the head
# output crosses the bus — GPU→CPU readback is DirectML's single most
# expensive primitive. torch-directml's op coverage varies, so probe
# index_select once here, forward and backward separately, and let each
# consumer fall back to a full-tensor download when its direction is
# unsupported (correct either way, just slower). The tiny (B,3)/(B,) VALUE
# outputs never need gathering — they cross whole (16 bytes/row).
def _probe_device_gather(device):
    if str(device) == 'cpu':
        return True, True                 # plain aten ops — always available
    fwd = bwd = False
    try:
        x   = torch.arange(12.0, device=device).reshape(4, 3)
        idx = torch.tensor([2, 0, 3], dtype=torch.int64, device=device)
        y   = x.index_select(0, idx).cpu()
        fwd = bool(torch.equal(y, torch.tensor([[6., 7., 8.],
                                                [0., 1., 2.],
                                                [9., 10., 11.]])))
    except Exception:
        pass
    try:
        x   = torch.ones(4, 3, device=device, requires_grad=True)
        idx = torch.tensor([1, 1, 3], dtype=torch.int64, device=device)
        y   = x.index_select(0, idx)
        y.backward(torch.ones_like(y))    # backward of gather = index_add
        g   = x.grad.detach().cpu().sum(dim=1)
        bwd = bool(torch.allclose(g, torch.tensor([0., 6., 0., 3.])))
    except Exception:
        pass
    return fwd, bwd

_GATHER_FWD_OK, _GATHER_BWD_OK = _probe_device_gather(DEVICE)
if str(DEVICE) != 'cpu':
    print(f'on-device gather: forward '
          f'{"OK" if _GATHER_FWD_OK else "UNAVAILABLE (full-download fallback)"}'
          f', backward '
          f'{"OK" if _GATHER_BWD_OK else "UNAVAILABLE (full-download fallback)"}')


# ── Network modules (identical to the sibling notebooks) ──────────────────────

class _GroupNorm(nn.Module):
    """GroupNorm from elementwise ops — DirectML-safe (the fused kernel's
    backward is broken there) and keeps NO running stats (train == eval)."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.num_groups = num_groups
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        n, c = x.shape[0], x.shape[1]
        xg = x.reshape(n, self.num_groups, -1)
        mean = xg.mean(dim=2, keepdim=True)
        var = (xg - mean).pow(2).mean(dim=2, keepdim=True)
        xg = (xg - mean) / torch.sqrt(var + self.eps)
        x = xg.reshape(x.shape)
        return x * self.weight.view(1, c, 1, 1) + self.bias.view(1, c, 1, 1)


def _norm(channels):
    groups = min(8, channels)
    while channels % groups != 0:
        groups -= 1
    return _GroupNorm(groups, channels)


def _softplus(x):
    """Numerically-stable softplus from elementary ops only. torch's fused
    aten::softplus has no DirectML kernel; relu/abs/exp/log/add all do.
    Identical to F.softplus: log(1+e^x) = max(x,0) + log(1 + e^-|x|)."""
    return torch.relu(x) + torch.log(1.0 + torch.exp(-torch.abs(x)))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-GN-ReLU-Conv-GN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class ThompsonHybridChessNet(nn.Module):
    """
    Two heads on one shared trunk:
      VALUE  : (p_win, p_draw, p_loss) + confidence for the CURRENT position
               (mover's perspective) — the thompson_value head, ~4 outputs.
      POLICY : logits over the 4674 actions + a scalar concentration β — a
               Dirichlet-Categorical belief over which legal move is best. A
               flat Linear policy head (gathered to legal entries at every
               boundary, so its 4674-wide output never crosses a bus) rather
               than a spatial from-square head, for robustness — the spatial
               head's index remap is a documented, bug-prone follow-up.

    Input  : (B, 20, 8, 8) — chess's native observation planes
    Body   : Conv stem → N × ResBlock(channels, SE)
    Head   : 1×1 conv (H ch) → flatten (H·64) →
               v_dist Linear(·, 3) → softmax → (p_win, p_draw, p_loss)
               v_conf Linear(·, 1) → softplus → α₀ ∈ [CONF_MIN, CONF_MAX]
               p_log  Linear(·, NUM_ACTIONS) → policy logits
               p_conf Linear(·, 1) → softplus → β ∈ [POL_CONF_MIN, POL_CONF_MAX]

    forward(x) → (v_probs (B,3), v_conf (B,), p_logits (B,A), p_conf (B,)).
    """
    _HEAD_CH = 8

    def __init__(self, channels=64, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(_OBS_SHAPE[0], channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Conv2d(channels, self._HEAD_CH, 1, bias=False),
            _norm(self._HEAD_CH),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        flat = self._HEAD_CH * _OBS_SHAPE[1] * _OBS_SHAPE[2]
        self.v_dist = nn.Linear(flat, 3)
        self.v_conf = nn.Linear(flat, 1)
        self.p_log  = nn.Linear(flat, _NUM_ACTIONS)
        self.p_conf = nn.Linear(flat, 1)
        # SIGMOID-bounded confidence (not softplus+clamp): the clamp form has
        # ZERO gradient once the pre-clamp value exceeds the cap, so confidence
        # saturation is a one-way door — the first hybrid run pinned α₀ at the
        # cap within ~150 episodes and it could never come back down. A sigmoid
        # maps to (MIN, MAX) with a gradient that never dies, so the head can
        # settle at whatever finite level the (now honest — see the split-z
        # loss) targets support. Bias-init to a LOW starting confidence so
        # search dominates from generation 0.
        nn.init.constant_(self.v_conf.bias, -3.9)  # CONF_MIN+(CONF_MAX-·)σ(-3.9)≈2.4
        nn.init.constant_(self.p_conf.bias, -6.6)  # POL_CONF_MIN+(·)σ(-6.6)≈2.3

    def forward(self, x):
        h       = self.head(self.body(self.stem(x)))
        v_probs = F.softmax(self.v_dist(h), dim=-1)
        v_conf  = CONF_MIN + (CONF_MAX - CONF_MIN) * torch.sigmoid(
            self.v_conf(h).squeeze(-1))
        p_logits = self.p_log(h)
        p_conf  = POL_CONF_MIN + (POL_CONF_MAX - POL_CONF_MIN) * torch.sigmoid(
            self.p_conf(h).squeeze(-1))
        return v_probs, v_conf, p_logits, p_conf


print(f'Device: {DEVICE}')
_demo = ThompsonHybridChessNet()
_np = sum(p.numel() for p in _demo.parameters())
_pol = sum(p.numel() for p in _demo.p_log.parameters())
print(f'ThompsonHybridChessNet params: {_np:,} '
      f'(policy head {_pol:,} = {100.0*_pol/_np:.0f}%)')
del _demo

In [ ]:
# ── Training-sample format ─────────────────────────────────────────────────────
# A sample carries BOTH heads' targets for one position:
#   obs fp16 (1280,)      — the position (mover's perspective)
#   v_counts fp32 (3,)    — VALUE target: posterior (win,draw,loss) pseudo-counts
#   legal_idx int32 (m,)  — the position's legal actions (for the policy softmax
#                           denominator; the policy head is gathered to these)
#   visit_counts fp32(m,) — POLICY target: MCTS visit counts per legal action
#                           (the search's belief about which move is best)
#   solved bool           — solver-labeled (exact ground truth) — diagnostic
#   on_path bool          — the position lies on the actually-played trajectory
#                           (z-mixed + prioritized in replay) vs a solver aux
# Value is dense-tiny (3 floats); the policy target is sparse over legal moves
# (~35, not 4674) and reuses the tested gathered-loss machinery.

def augment_sample(obs_flat, v_counts, legal_idx, visit_counts,
                   solved=False, on_path=True, z_col=-1):
    """No-op wrapper (chess has no usable board symmetry — see the intro),
    returning the single sample in a list for a call-site identical to the
    sibling notebooks. `z_col` is the game-outcome column (0/1/2 from the
    mover's perspective) for on-path finished-game samples, or -1 (no
    outcome) — see resolve_z."""
    return [(np.asarray(obs_flat,      dtype=np.float16),
             np.asarray(v_counts,      dtype=np.float32),
             np.asarray(legal_idx,     dtype=np.int32),
             np.asarray(visit_counts,  dtype=np.float32),
             bool(solved), bool(on_path), int(z_col))]


# ── Hybrid Bayesian-MCTS tree ─────────────────────────────────────────────────
# The value-belief machinery (value-grid Dirichlet PMFs, P(best) mixture
# propagation, MCTS-Solver, innovation calibration) is identical to the
# thompson_z / thompson_value notebooks — see those for the full derivations.
# What is NEW is the SELECTION rule and node expansion:
#
#   * Expanding a node is ONE forward pass on the node's OWN state, giving a
#     value belief V(s) (win/draw/loss) AND a policy prior P(a) over the legal
#     moves (+ concentration β). No child is evaluated at expansion, so one
#     simulation opens one node — the tree can go DEEP (the thompson_value
#     design paid ~35 child evals to open a single node, capping depth).
#
#   * Two-stage Thompson selection at a node:
#       - OPENED edges (a child exists, or the edge is a proven terminal) each
#         carry a GROUNDED value belief; Thompson-sample each.
#       - UNOPENED edges share no value claim (we haven't looked). The POLICY
#         prior picks ONE candidate a* among them by Thompson-sampling the
#         Dirichlet-Categorical belief Dir(β·P) — a real posterior draw, the
#         Bayesian analogue of PUCT's c_puct·P·√N/(1+n) term with no tuned
#         constant. Its competing value is one FPU draw from the node's OWN
#         value belief V(s) (first-play-urgency = "a fresh move keeps me about
#         as well off as I am here" — an honest prior over a seen state's
#         value, never a fabricated per-action value).
#       - argmax over {opened edges' samples} ∪ {a*'s FPU sample} decides
#         whether to deepen an existing line or open a new move.
#     As edges open the policy stops mattering (no unopened edges) and this is
#     pure thompson_z value-belief search. The node's propagated VALUE is the
#     mixture over OPENED edges only (unopened edges never inflate it — that
#     would reintroduce the maximization bias mixture propagation exists to
#     avoid), falling back to V(s) when nothing is opened yet.
#
# So the value head is thompson_value's (grounded, never fabricated, tiny) and
# the policy head does exactly the job AlphaZero's policy proved it does
# (prioritize a narrow deep search) — but as a Bayesian ordering belief the
# search Thompson-samples, not a heuristic.

_GRID_G  = 33
_GRID_X  = (np.arange(_GRID_G) + 0.5) / _GRID_G          # (G,) unit-interval grid
_GRID_V  = 2.0 * _GRID_X - 1.0                            # (G,) value-space read-out
_GRID_V2 = _GRID_V ** 2
_GRID_LX  = np.log(_GRID_X)
_GRID_L1X = np.log1p(-_GRID_X)
_SPIKE   = np.eye(_GRID_G)                               # _SPIKE[j] = onehot at j
_SPIKE_WIN, _SPIKE_LOSS, _SPIKE_DRAW = (_SPIKE[_GRID_G - 1], _SPIKE[0],
                                        _SPIKE[_GRID_G // 2])
_VLOSS_PEN = 1.0      # within-wave virtual-loss penalty on a sampled edge value
_MAX_FRAC  = 0.0      # 0 = pure mixture propagation; 1 = hard max
# The policy prior must PERSISTENTLY steer selection (not just first-open
# order), or a flat early policy leaves the search uniform and its visit-count
# target is noise — the failure mode that stalled the first hybrid run. A PUCT
# bonus U = c_puct·P(a)·sqrt(ΣN)/(1+N(a)) added to each edge's Thompson value
# sample keeps high-prior moves visited until their own value evidence takes
# over (AlphaZero's proven exploration term; the Thompson draw still carries
# the value uncertainty, so there is no exploration constant on THAT).
_C_PUCT = 1.5
_FPU_REDUCTION = 0.25    # pessimize unvisited moves so search exploits known ones
_POLICY_CONC_CAP = 40.0  # cap the policy visit-target concentration (keeps the
                         # policy loss on the same scale as the value loss, and
                         # β from being trained toward the raw 100-400 sim count)
# One-edge-at-a-time expansion almost never fully proves an interior node
# (aux≈0), which starved BOTH the solver-label training signal AND the
# calibration's ground-truth channel. Remedy: at a LOW-branching (forcing) node
# — few legal moves, i.e. in check or a sparse endgame, exactly where terminals
# and proofs live — apply every move and mark the terminal children proven.
# This needs NO NN eval (terminals need none), costs only k clones where k is
# small, and recovers thompson_value's dense terminal detection where it
# matters. High-branching nodes are skipped (rarely have terminal children; the
# clones would be wasted). A generous cap because the restart curriculum seeds
# games near endings, so most self-play roots ARE low-branching.
_FORCING_MAXLEGAL = 12


# ── Innovation-based confidence calibration (empirical Bayes — NO caps) ───────
# Unchanged from the sibling notebooks: model VALUE-head miscalibration as a
# variance-inflation factor λ (true error variance = λ × claimed), estimated
# from backup innovations (an edge's held belief vs its one-ply-deeper
# replacement, and vs terminal ground truth), scored with a bounded-influence
# t-statistic normalized so a calibrated net gives λ → 1. Applied at value-
# belief construction, scaling Var[v].
_TDOF    = 4.0
_VAR_RES = (2.0 / _GRID_G) ** 2 / 12.0

def _psi(z2):
    return (_TDOF + 1.0) * z2 / (_TDOF + z2)

_zg = np.linspace(-10.0, 10.0, 8001)          # Riemann sum on a uniform grid
_PSI_NORM = float(np.sum(_psi(_zg ** 2)      # (np.trapz was removed in numpy 2)
                         * np.exp(-0.5 * _zg ** 2) / np.sqrt(2.0 * np.pi))
                  * (_zg[1] - _zg[0]))
del _zg


class _Calib:
    """Running empirical-Bayes estimate of the variance-inflation factor λ."""
    __slots__ = ('s', 'n', 'd')

    def __init__(self, prior_n=50.0, halflife=2000.0):
        self.s = prior_n
        self.n = prior_n
        self.d = 0.5 ** (1.0 / halflife)

    def observe(self, e2, var_sum):
        z2 = e2 / (var_sum + 2.0 * _VAR_RES)
        self.s = self.s * self.d + _psi(z2) / _PSI_NORM
        self.n = self.n * self.d + 1.0

    def lam(self):
        return self.s / self.n


def _beta_pmf_rows(alpha, beta):
    """Discretize Beta(alpha_i, beta_i) onto the unit grid → (k, G) PMF rows."""
    logw = ((alpha[:, None] - 1.0) * _GRID_LX[None, :]
            + (beta[:, None] - 1.0) * _GRID_L1X[None, :])
    logw -= logw.max(axis=1, keepdims=True)
    w = np.exp(logw)
    return w / w.sum(axis=1, keepdims=True)


def _flip_pmf(pmf):
    """Reflect a value PMF around v=0 (v -> -v)."""
    return pmf[::-1].copy()


def _dirichlet_leaf_belief(alpha_w, alpha_d, alpha_l, lam=1.0):
    """Dirichlet(alpha_w, alpha_d, alpha_l) → (v-belief PMF (k,G), mean draw
    (k,)) via the exact independent (q,d) decomposition (verified numerically
    in the sibling notebooks), moment-matching a Beta to the closed-form
    (E[v], Var[v]). `lam` scales the claimed Var[v] to the observed error
    scale before the belief is built."""
    C = alpha_w + alpha_d + alpha_l
    Ed = alpha_d / C
    Vd = alpha_d * (alpha_w + alpha_l) / (C**2 * (C + 1.0))
    Eq = alpha_w / (alpha_w + alpha_l)
    Vq = alpha_w * alpha_l / ((alpha_w + alpha_l)**2 * (alpha_w + alpha_l + 1.0))
    EX, VX = 1.0 - Ed, Vd
    EY, VY = 2.0 * Eq - 1.0, 4.0 * Vq
    Ev = EX * EY
    Vv = EX**2 * VY + EY**2 * VX + VX * VY
    mu01  = (Ev + 1.0) / 2.0
    var01 = np.clip(Vv / 4.0 * lam, 1e-9, None)
    var01 = np.minimum(var01, mu01 * (1.0 - mu01) * 0.999)
    conc  = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    a_beta = np.maximum(mu01 * conc, ALPHA_FLOOR)
    b_beta = np.maximum((1.0 - mu01) * conc, ALPHA_FLOOR)
    return _beta_pmf_rows(a_beta, b_beta), Ed


def _prob_best(E, C):
    """P(edge i is the argmax) for each edge, from PMFs `E` and their inclusive
    CDFs `C`: sum_x p_i(x) * prod_{j!=i} P(X_j < x), tie-split, log-space LOO."""
    logG = np.log(np.clip(C - 0.5 * E, 1e-12, 1.0))
    loo  = np.exp(logG.sum(axis=0)[None, :] - logG)
    w = (E * loo).sum(axis=1)
    s = w.sum()
    return w / s if s > 0 else np.full(len(w), 1.0 / len(w))


class _TNode:
    """One expanded state.

    `vself`/`dself`   the value head's own belief for THIS state (mover's
                      perspective) — the FPU prior for unopened edges and the
                      node's fallback value when nothing is opened yet.
    `edge[i]`/`draw[i]` the GROUNDED value belief for edge i, valid only once
                      the edge is opened (a child exists) or proven (term>=0);
                      initialized to the FPU prior and ignored until then.
    `pol[i]`          the policy head's P(action i is best) over the legal
                      moves; `beta` its scalar concentration. Ordering only.
    `visits[i]`       how many simulations selected edge i (the policy target).
    `term[i] >= 0`    a PROVEN outcome for edge i (_WIN/_DRAW/_LOSS).
    """
    __slots__ = ('player', 'legal', 'vself', 'dself', 'edge', 'draw', 'pol',
                 'beta', 'visits', 'vloss', 'term', 'children', 'obs')

    def __init__(self, player, legal, v_probs, v_conf, pol, beta, lam=1.0):
        self.player = player
        self.legal  = np.asarray(legal, dtype=np.int32)
        self.obs    = None
        k = len(self.legal)
        vp = np.asarray(v_probs, dtype=np.float64)
        cf = float(v_conf)
        aw = max(cf * vp[0], ALPHA_FLOOR)
        ad = max(cf * vp[1], ALPHA_FLOOR)
        al = max(cf * max(vp[2], 0.0), ALPHA_FLOOR)
        vpmf, dd = _dirichlet_leaf_belief(np.array([aw]), np.array([ad]),
                                          np.array([al]), lam)
        self.vself = vpmf[0]
        self.dself = float(dd[0])
        self.edge  = np.tile(self.vself, (k, 1))       # FPU until opened
        self.draw  = np.full(k, self.dself)
        self.pol   = np.asarray(pol, dtype=np.float64)
        self.beta  = float(beta)
        self.visits   = np.zeros(k, dtype=np.int64)
        self.vloss    = np.zeros(k, dtype=np.int32)
        self.term     = np.full(k, -1, dtype=np.int8)
        self.children = [None] * k


_TERM_PMF  = np.stack([_SPIKE_WIN, _SPIKE_DRAW, _SPIKE_LOSS])
_TERM_DRAW_VAL = np.array([0.0, 1.0, 0.0])


def _set_term(node, idx, outcome):
    node.term[idx] = outcome
    node.edge[idx] = _TERM_PMF[outcome]
    node.draw[idx] = _TERM_DRAW_VAL[outcome]


def _opened_mask(node):
    """Edges with a GROUNDED value belief: a child exists, or the edge is a
    proven terminal. Unopened edges (FPU prior only) are excluded from the
    node's propagated value."""
    m = (node.term >= 0)
    for i, c in enumerate(node.children):
        if c is not None:
            m[i] = True
    return m


def _node_beliefs(node):
    """The node's own (value PMF, mean draw) = the P(best)-weighted mixture
    over its OPENED edges (Tesauro mixture propagation). Falls back to the
    value head's own belief when no edge is opened yet."""
    mask = _opened_mask(node)
    if not mask.any():
        return node.vself.copy(), node.dself
    E = node.edge[mask]
    D = node.draw[mask]
    if E.shape[0] == 1:
        return E[0].copy(), float(D[0])
    C = np.cumsum(E, axis=1)
    np.clip(C, 0.0, 1.0, out=C)
    w = _prob_best(E, C)
    mix = w @ E
    mean_draw = float(w @ D)
    if _MAX_FRAC > 0.0:
        Fmax = C.prod(axis=0)
        mx = np.empty(_GRID_G)
        mx[0]  = Fmax[0]
        mx[1:] = np.diff(Fmax)
        np.maximum(mx, 0.0, out=mx)
        ms = mx.sum()
        mix = (1.0 - _MAX_FRAC) * mix + _MAX_FRAC * (mx / ms if ms > 0 else mix)
    s = mix.sum()
    return (mix / s if s > 0 else np.full(_GRID_G, 1.0 / _GRID_G)), mean_draw


def _sample_pmf(pmf, u):
    """Inverse-CDF sample of a value from one PMF given a uniform u."""
    c = np.cumsum(pmf)
    idx = min(int((c < u).sum()), _GRID_G - 1)
    return _GRID_V[idx]


def _choose_edge(node, rng, exclude=None):
    """One PUCT-style Thompson selection → the chosen edge index. Every edge's
    value belief is Thompson-sampled (unopened edges hold the FPU prior — the
    node's own value belief — pessimized by _FPU_REDUCTION; proven edges hold
    exact spikes), and a PERSISTENT policy-prior bonus
    U = c_puct·P(a)·sqrt(ΣN)/(1+N(a)) is added, decaying as an edge accrues
    visits. The Thompson draw carries the value uncertainty (no exploration
    constant on it); U carries policy-guided exploration, so a high-prior move
    keeps getting tried until its own value evidence takes over — the
    AlphaZero behavior the earlier two-stage rule lost the moment every edge
    was opened. `exclude` holds edge indices already being opened THIS wave
    (kept diverse), relaxed if it would leave no candidate."""
    C = np.cumsum(node.edge, axis=1)
    u = rng.random_sample(len(node.legal))
    gi = np.minimum((C < u[:, None]).sum(axis=1), _GRID_G - 1)
    v = _GRID_V[gi].copy()
    opened = _opened_mask(node)
    if not opened.all():
        v[~opened] -= _FPU_REDUCTION
    N = node.visits.astype(np.float64)
    U = _C_PUCT * node.pol * np.sqrt(N.sum() + 1.0) / (1.0 + N)
    score = v + U - _VLOSS_PEN * node.vloss
    if exclude:
        masked = score.copy()
        for i in exclude:
            if 0 <= i < len(masked):
                masked[i] = -np.inf
        if np.isfinite(masked).any():
            score = masked
    return int(score.argmax())


def _select_leaf(root, root_state, rng, calib=None, exclude=None):
    """Descend by two-stage Thompson selection until an unopened edge (→ open
    it: apply the move, return the leaf state for evaluation) or a proven/
    terminal edge (→ nothing to eval). Applies virtual loss and increments the
    visit count along the path. A NEWLY-opened edge whose child is terminal is
    proven immediately (and, for `calib`, scored against the ±1/0 ground
    truth). `exclude` is the set of (id(node), idx) unopened edges already
    being opened this wave."""
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        ex = None
        if exclude:
            ex = {i for (nid, i) in exclude if nid == id(node)}
        idx = _choose_edge(node, rng, ex)
        node.vloss[idx] += 1
        node.visits[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        child = node.children[idx]
        if child is None:                       # unopened → open it
            state.apply_action(int(node.legal[idx]))
            if state.is_terminal():
                r = state.returns()[node.player]
                if calib is not None:
                    m0 = float(node.edge[idx] @ _GRID_V)
                    v0 = max(float(node.edge[idx] @ _GRID_V2) - m0 * m0, 0.0)
                    calib.observe((m0 - r) ** 2, v0)
                _set_term(node, idx,
                          _WIN if r > 0 else (_LOSS if r < 0 else _DRAW))
                return path, None, None
            return path, state, (node, idx)
        state.apply_action(int(node.legal[idx]))
        node = child


def _backup(path):
    """Remove virtual loss and recompute each OPENED edge on the path from its
    child's current value (bottom-up); value flips sign up a ply, draw does
    not. Unopened edges keep their FPU prior (untouched). Nothing accumulates."""
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        if node.term[idx] < 0 and node.children[idx] is not None:
            vpmf, mdraw = _node_beliefs(node.children[idx])
            node.edge[idx] = _flip_pmf(vpmf)
            node.draw[idx] = mdraw


def _edge_innovation(node, idx, m0, v0, calib):
    """First-open innovation for a freshly opened edge, read AFTER its backup:
    the edge held the FPU prior (m0, v0); _backup replaced it with the flipped
    child value (or a proof spike). Scores the value head's calibration."""
    p = node.edge[idx]
    m1 = float(p @ _GRID_V)
    v1 = max(float(p @ _GRID_V2) - m1 * m1, 0.0)
    calib.observe((m0 - m1) ** 2, v0 + v1)


# ── MCTS-Solver ───────────────────────────────────────────────────────────────

def _node_solved_outcome(node):
    """Proven outcome for node.player if solved, else None: WIN as soon as any
    edge is a proven win; else DRAW if any edge is a proven draw and all edges
    are proven; LOSS only once every edge is a proven loss."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _solver_sweep(path, aux=None, conf_cap=None):
    """Walk the path deepest-edge-first; prove each edge whose child has become
    solved (flipped — the parent's mover is the opponent), stopping at the
    first unsolved child or already-proven edge. Emits each NEWLY-solved node
    once as a solver-labeled sample (child.obs, its value + policy targets)
    into `aux` — exact game-theoretic labels, deduped by the once-per-node
    parent-edge transition."""
    for k in range(len(path) - 1, -1, -1):
        parent, idx = path[k]
        child = parent.children[idx]
        if child is None:
            break
        out = _node_solved_outcome(child)
        if out is None or parent.term[idx] >= 0:
            break
        _set_term(parent, idx, int(_FLIP_TERM[out]))
        if aux is not None and child.obs is not None:
            aux.append((child.obs, state_value_target(child, conf_cap))
                       + policy_target(child))


def state_value_target(node, conf_cap):
    """Node posterior VALUE belief → dense (3,) fp32 counts (win,draw,loss):
    the mixture belief moment-matched back to proportions × concentration
    (from Var[v], capped at conf_cap), or the exact spike for a solved node."""
    out = _node_solved_outcome(node)
    if out is not None:
        cnt = np.full(3, ALPHA_FLOOR, dtype=np.float32)
        cnt[out] = conf_cap - 2.0 * ALPHA_FLOOR
        return cnt
    vpmf, d = _node_beliefs(node)
    mean_v = float(vpmf @ _GRID_V)
    var_v  = max(float(vpmf @ _GRID_V2) - mean_v * mean_v, 1e-9)
    mu01   = (mean_v + 1.0) / 2.0
    var01  = max(var_v / 4.0, 1e-9)
    conc   = max(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    conc   = min(conc, conf_cap)
    nondraw = min(max(1.0 - d, 0.0), 1.0)
    p_win  = min(max((mean_v + nondraw) / 2.0, 0.0), nondraw)
    p_loss = nondraw - p_win
    return (np.array([p_win, d, p_loss]) * conc).astype(np.float32)


def policy_target(node):
    """Node POLICY target: (legal_idx int32 (k,), visit_counts fp32 (k,)) over
    ALL legal moves (zeros for never-selected edges — the full legal set is
    needed for the training softmax denominator). The visit distribution is
    the search's belief about which move is best; the policy head is trained
    to it as a Dirichlet-Categorical (the same evidence loss as the value
    head, over m legal categories). A solved node teaches the proven move: all
    visit mass on its best edge, so the policy learns to find forced lines. The
    visit counts are capped at _POLICY_CONC_CAP (mean-preserving) so the policy
    loss stays on the same scale as the value loss and β is not trained toward
    the raw 100-400 sim count."""
    out = _node_solved_outcome(node)
    if out is not None:
        vc = np.full(len(node.legal), 0.0, dtype=np.float32)
        vc[_solved_best_edge(node, out)] = _POLICY_CONC_CAP
        return node.legal.astype(np.int32), vc
    vis = node.visits.astype(np.float64)
    tot = vis.sum()
    if tot > _POLICY_CONC_CAP:
        vis = vis * (_POLICY_CONC_CAP / tot)           # mean-preserving cap
    return node.legal.astype(np.int32), vis.astype(np.float32)


def _solved_best_edge(node, out):
    """The edge realizing a solved node's outcome: the proven win (mover plays
    it), else for a proven draw the proven-draw edge, else any (all lost)."""
    if out == _WIN:
        return int(np.nonzero(node.term == _WIN)[0][0])
    if out == _DRAW:
        return int(np.nonzero(node.term == _DRAW)[0][0])
    return int(node.visits.argmax())


def resolve_z(samples, returns):
    """Attach the game-outcome column z_col (0/1/2 from each sample's mover's
    perspective) to every on-path sample — WITHOUT blending it into the value
    counts. z is a SINGLE noisy draw of the outcome; the earlier design
    averaged it into the count vector, which laundered away that variance and
    let the evidence loss treat an extreme mean as high-concentration truth
    (the confidence runaway). Instead z is scored in training as a separate
    single-draw likelihood (a cross-entropy on the value MEAN, independent of
    the predicted concentration — see the training cell), so a net that stays
    confident pays full price every time a game's outcome surprises it, and
    outcome variance bounds confidence with no cap needed. The value COUNTS
    stay the pure search posterior, whose honest search-resolved concentration
    is what trains α₀. Proven played edges get z_col=-1 (their exact solver
    label is already the target). Returns
    (obs, v_counts, legal_idx, visit_counts, z_col)."""
    out = []
    for obs, vc, li, vis, player, proven in samples:
        if proven:
            z = -1
        else:
            r = returns[player]
            z = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
        out.append((obs, vc, li, vis, z))
    return out


def strip_episode_meta(samples):
    """Ply-capped games: drop the metadata, z_col=-1 (outcome never observed)."""
    return [(o, vc, li, vis, -1) for o, vc, li, vis, _p, _v in samples]


# ── Backward-restart curriculum (unchanged from the sibling notebooks) ────────

def restart_prefix(seq, rng, k_min, k_max):
    """Prefix of a stored action sequence to replay for a restart: drop a
    random k ∈ [k_min, k_max] tail plies. None if too short."""
    if len(seq) <= k_min:
        return None
    k = int(rng.randint(k_min, min(k_max, len(seq) - 1) + 1))
    return seq[:len(seq) - k]


def _root_value(node):
    vpmf, _ = _node_beliefs(node)
    return float(vpmf @ _GRID_V)


def _descend(root, action):
    """Tree reuse: the played action's subtree becomes the next search's root.
    None when the edge was never opened."""
    if root is None:
        return None
    hit = np.nonzero(root.legal == action)[0]
    return root.children[int(hit[0])] if len(hit) else None


def root_pick(root, rng, thompson):
    """Commit a root move from the search statistics. VISIT-BASED
    (AlphaZero-standard), NOT value-mean-based: the visit count integrates the
    value posterior over every simulation, so it is robust to a single lucky
    Thompson draw on a barely-visited edge — the failure mode that made a
    weak/uncertain value head commit rim/rook moves and march in board order.
    The Thompson value posterior still drives the SEARCH itself (edge
    selection in `_choose_edge`); this is only the final commit, which
    AlphaZero always takes from visit counts.
      thompson=True  (temperature phase): sample a move ∝ its visit count, for
                     opening/self-play diversity.
      thompson=False (greedy — eval + post-temperature self-play): play the
                     most-visited move; a proven win is played outright; ties
                     break by posterior mean value, then at random.
    If nothing was opened (0-sim degenerate) fall back to the policy argmax."""
    opened = _opened_mask(root)
    if not opened.any():
        return int(root.legal[int(root.pol.argmax())])
    if thompson:
        w = root.visits.astype(np.float64)
        s = w.sum()
        if s <= 0:
            return int(root.legal[int(root.pol.argmax())])
        i = int((np.cumsum(w / s) < rng.random_sample()).sum())
        return int(root.legal[min(i, len(root.legal) - 1)])
    wins = np.nonzero(root.term == _WIN)[0]
    pool = wins if len(wins) else np.nonzero(opened)[0]
    vis = root.visits[pool]
    cand = pool[vis == vis.max()]
    if len(cand) > 1:
        vmean = root.edge[cand] @ _GRID_V
        cand = cand[vmean >= vmean.max() - 1e-9]
        i = int(cand[rng.randint(len(cand))])
    else:
        i = int(cand[0])
    return int(root.legal[i])


# ── Node expansion: ONE forward pass on the node's own state ──────────────────

def _softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


def _scan_terminal_children(node, state, calib=None):
    """Low-branching-node solver signal: apply every legal move once and mark
    the edges whose child is terminal as PROVEN (no NN eval — terminals need
    none). Lets MCTS-Solver prove forcing lines and emits solver-label training
    samples where thompson_value's full expansion would; when `calib` is given,
    each terminal is also a GROUND-TRUTH innovation (the node's prior value
    belief for that edge vs the actual ±1/0), the signal the calibration was
    starved of. Only called for nodes with ≤ _FORCING_MAXLEGAL legal moves."""
    me = node.player
    for i, a in enumerate(node.legal):
        ch = state.clone()
        ch.apply_action(int(a))
        if ch.is_terminal():
            r = ch.returns()[me]
            if calib is not None:
                m0 = float(node.edge[i] @ _GRID_V)
                v0 = max(float(node.edge[i] @ _GRID_V2) - m0 * m0, 0.0)
                calib.observe((m0 - r) ** 2, v0)
            _set_term(node, i, _WIN if r > 0 else (_LOSS if r < 0 else _DRAW))


def _node_from_eval(state, v_probs, v_conf, p_logits_legal, p_conf, lam,
                    calib=None):
    """Build a _TNode from one network evaluation of `state`: the value head's
    (win,draw,loss)+conf and the policy head's logits GATHERED to the legal
    moves (+concentration). The policy mean is the softmax over legal logits.
    At a low-branching node, terminal children are detected and proven now."""
    legal = state.legal_actions()
    pol   = _softmax(np.asarray(p_logits_legal, dtype=np.float64))
    node  = _TNode(state.current_player(), legal, v_probs, float(v_conf),
                   pol, float(p_conf), lam)
    node.obs = np.asarray(state.observation_tensor(state.current_player()),
                          dtype=np.float16)
    if len(legal) <= _FORCING_MAXLEGAL:
        _scan_terminal_children(node, state, calib)
    return node


def _expand_state_node(network, device, state, lam=1.0, calib=None):
    """Expand one state into a _TNode with a single forward pass (used for
    search roots and small-batch callers; the self-play loops batch many
    expansions into one pass instead)."""
    x = state_to_tensor(state, device)
    with torch.no_grad():
        vp, vc, pl, pc = network(x)
    legal = state.legal_actions()
    pl_legal = pl[0].cpu().numpy()[legal]
    return _node_from_eval(state, vp[0].cpu().numpy(), float(vc[0]),
                           pl_legal, float(pc[0]), lam, calib)


class ThompsonMCTSBot:
    """mcts_search(state) → root _TNode. Each wave selects up to `batch_size`
    leaves (each opening ONE node), evaluates their states in one NN forward
    pass, wires them in, backs up, and runs the solver sweep. Weights are read
    live from `network`."""

    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, random_state=None, calib=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self._rng            = random_state or np.random.RandomState()
        self.calib           = calib if calib is not None else _Calib()

    def mcts_search(self, state):
        root = _expand_state_node(self.network, self.device, state,
                                  self.calib.lam(), self.calib)
        sims = 0
        while sims < self.max_simulations:
            if _node_solved_outcome(root) is not None:
                break
            wave    = min(self.batch_size, self.max_simulations - sims)
            pending = []                 # (path, (node, idx)) awaiting eval
            opening = set()              # (id(node), idx) opened this wave
            evals   = {}                 # (id(node), idx) -> (node, idx, state)
            for _ in range(wave):
                path, st, edge = _select_leaf(root, state, self._rng,
                                              self.calib, opening)
                if st is None:           # proven/terminal edge
                    _backup(path); _solver_sweep(path); sims += 1
                else:
                    node, idx = edge
                    opening.add((id(node), idx))
                    pending.append((path, edge))
                    evals.setdefault((id(node), idx), (node, idx, st))
            # Unique leaf states → one forward pass; wire the children in.
            pre = {}
            if evals:
                items = list(evals.values())
                x = batch_to_tensor([st.observation_tensor(st.current_player())
                                     for _, _, st in items], self.device)
                with torch.no_grad():
                    vp, vc, pl, pc = self.network(x)
                vp = vp.cpu().numpy(); vc = vc.cpu().numpy()
                pl = pl.cpu().numpy();  pc = pc.cpu().numpy()
                lam = self.calib.lam()
                for j, (node, idx, st) in enumerate(items):
                    legal = st.legal_actions()
                    m0 = float(node.edge[idx] @ _GRID_V)
                    v0 = max(float(node.edge[idx] @ _GRID_V2) - m0 * m0, 0.0)
                    pre[(id(node), idx)] = (node, idx, m0, v0)
                    node.children[idx] = _node_from_eval(
                        st, vp[j], float(vc[j]), pl[j][legal], float(pc[j]),
                        lam, self.calib)
            # Mixture-propagate each fresh leaf up its own path.
            for path, (node, idx) in pending:
                _backup(path); sims += 1
            # Solver proofs (deepest edge of each path) + calibration.
            for path, (node, idx) in pending:
                _solver_sweep(path)
            for node, idx, m0, v0 in pre.values():
                _edge_innovation(node, idx, m0, v0, self.calib)
        return root


def make_thompson_bot(game, network, device, num_simulations, batch_size=16):
    return ThompsonMCTSBot(game, network, device, num_simulations,
                           batch_size=batch_size)


# ── Evaluation ─────────────────────────────────────────────────────────────────

def policy_action(network, state, device, sample=False, rng=None):
    """Search-free move from the POLICY head: one forward pass, gather the
    legal logits, take the argmax (or, sample=True, one Dirichlet-Categorical
    Thompson draw). Tests the ordering head directly."""
    with torch.no_grad():
        _vp, _vc, pl, pc = network(state_to_tensor(state, device))
    legal = state.legal_actions()
    logits = pl[0].cpu().numpy()[legal]
    p = _softmax(logits)
    if sample:
        rng = rng or np.random
        g = rng.standard_gamma(np.maximum(float(pc[0]) * p, 1e-9))
        return int(legal[int(g.argmax())])
    return int(legal[int(p.argmax())])


MAX_EVAL_PLIES = 300


def play_match(net_a, net_b, game, n_games, device):
    """net_a vs net_b over n_games, alternating colours. net == None ==> random.
    Search-free policy-head moves. Returns (wins_a, draws, wins_b)."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = policy_action(net, state, device, sample=True)
            state.apply_action(action)
            ply += 1
        if not state.is_terminal():
            dd += 1
            continue
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    return root_pick(bot.mcts_search(state), bot._rng, thompson=False)


import math
import itertools
from collections import defaultdict


# ── Unified eval pool: one shared Elo table over (checkpoint × sim-budget) ────
# Every checkpoint enters the pool as MULTIPLE players — one per eval sim budget
# (e.g. label@0 = search-free policy head, label@32, label@128) — plus a single
# 'random' mover. All share ONE Elo table, so search-vs-no-search and
# generation-vs-generation are directly comparable in the same ratings. A
# player is a dict {key, label, sims, net}.

def _eval_pick(player, state, device, bots):
    net = player['net']
    if net is None:
        return random.choice(state.legal_actions())
    if player['sims'] <= 0:
        return policy_action(net, state, device, sample=False)
    return _mcts_move(bots[player['key']], state)


def _play_eval_game(pa, pb, game, device, bots, elos, wdl, k, opening_plies):
    """One game pa (player 0) vs pb (player 1); update Elo + W/D/L in place."""
    state = game.new_initial_state()
    ply = 0
    while not state.is_terminal() and ply < MAX_EVAL_PLIES:
        if ply < opening_plies:
            action = random.choice(state.legal_actions())
        else:
            action = _eval_pick(pa if state.current_player() == 0 else pb,
                                state, device, bots)
        state.apply_action(action)
        ply += 1
    r  = state.returns()[0]
    s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
    ka, kb = pa['key'], pb['key']
    e0 = 1.0 / (1.0 + 10 ** ((elos[kb] - elos[ka]) / 400.0))
    elos[ka] += k * (s0 - e0)
    elos[kb] += k * ((1.0 - s0) - (1.0 - e0))
    cell = wdl[(ka, kb)]
    if r > 0:   cell[0] += 1
    elif r < 0: cell[2] += 1
    else:       cell[1] += 1


def run_eval_pool(players, elos, wdl, game, device, new_keys,
                  k=32.0, games_per_pair=2, random_mult=5, opening_plies=0):
    """Rate the pool. Each of the `new_keys` players plays `games_per_pair`
    games (alternating colours) against every player that ALREADY existed
    (new-vs-new is left to the random phase); then `random_mult * len(players)`
    extra games are played between uniformly-random distinct pairs over ALL
    players, to keep older ratings mixing. `elos` must hold a rating for every
    player key. Bots for search players are built once and reused."""
    keys = [p['key'] for p in players]
    by_key = {p['key']: p for p in players}
    bots = {}
    for p in players:
        if p['net'] is not None and p['sims'] > 0:
            bots[p['key']] = make_thompson_bot(
                game, p['net'], device, p['sims'],
                batch_size=max(1, min(p['sims'], 16)))
    new = set(new_keys)
    existing = [key for key in keys if key not in new]
    schedule = []
    half = games_per_pair // 2
    for nk in new_keys:
        for ek in existing:
            schedule += [(nk, ek)] * half + [(ek, nk)] * half
            schedule += [(nk, ek)] * (games_per_pair - 2 * half)
    if len(keys) >= 2:
        for _ in range(random_mult * len(keys)):
            a, b = random.sample(keys, 2)
            schedule.append((a, b))
    random.shuffle(schedule)
    for ka, kb in schedule:
        _play_eval_game(by_key[ka], by_key[kb], game, device, bots,
                        elos, wdl, k, opening_plies)

In [ ]:
# ── Parallel self-play: many games, one forward pass ────────────────────────────
# ThompsonMCTSBot batches leaf-opens within ONE search. The next multiplier is
# batching ACROSS games: run N games concurrently, collect every game's
# open/expansion wave, and evaluate ALL their leaf states in a single NN
# forward pass. Each simulation opens ONE node (one eval), so a wave batches
# n_parallel × wave rows — feed that with enough games to keep the GPU full.
import os


def _nn_eval(network, device, states):
    """States → (v_probs (B,3), v_conf (B,), p_logits (B,A), p_conf (B,), obs
    fp16 (B,1280)) in one forward pass. The fp16 observations are returned so
    callers can stash them on the expanded nodes."""
    obs16 = np.asarray([s.observation_tensor(s.current_player())
                        for s in states], dtype=np.float16)
    x = batch_to_tensor(obs16, device)
    with torch.no_grad():
        vp, vc, pl, pc = network(x)
    return (vp.cpu().numpy(), vc.cpu().numpy(), pl.cpu().numpy(),
            pc.cpu().numpy(), obs16)


class ThompsonParallelSelfPlay:
    """Interleaves the searches of n_parallel self-play games so their
    open/expansion waves share one NN forward pass. Weights are read live from
    `network`."""

    def __init__(self, game, network, device, n_parallel=8, wave_per_game=4,
                 fast_sims=100, full_sims=400, fast_prob=0.75, temp_threshold=20,
                 conf_cap=100.0, seed=None,
                 pool_prob=0.0, checkpoint_dir=None, channels=None,
                 num_blocks=None, max_selfplay_plies=400, z_mix=0.5,
                 z_gamma=1.0, restart_prob=0.0, restart_k_min=2,
                 restart_k_max=30, restart_pool_cap=128, swing_thresh=0.6,
                 random_pool_frac=0.5):
        self.game = game
        self.network = network
        self.device = device
        self.n_parallel = n_parallel
        self.wave = wave_per_game
        self.fast_sims = fast_sims
        self.full_sims = full_sims
        self.fast_prob = fast_prob
        self.temp_threshold = temp_threshold
        self.conf_cap = conf_cap
        self.max_selfplay_plies = max_selfplay_plies
        self.z_mix = z_mix
        self.z_gamma = z_gamma
        self._rng = np.random.RandomState(seed)
        self.restart_prob = restart_prob
        self.restart_k_min = restart_k_min
        self.restart_k_max = restart_k_max
        self.restart_pool_cap = restart_pool_cap
        self.swing_thresh = swing_thresh
        self.random_pool_frac = random_pool_frac
        self._restart_pool = []
        self.pool_prob = pool_prob
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}
        self.calib = _Calib()
        self.last_lam = self.calib.lam()
        self.last_aux = 0
        self.last_restart_pool = 0
        self.stats = {'games': 0, 'draw': 0, 'cutoff': 0, 'decisive': 0,
                      'plies': 0}
        self.slots = [self._new_game() for _ in range(n_parallel)]

    def _push_seed(self, seq):
        if len(seq) >= 2:
            self._restart_pool.append(list(seq))
            if len(self._restart_pool) > self.restart_pool_cap:
                del self._restart_pool[0]

    def _load_pool_net(self, label):
        net = self._pool_nets.get(label)
        if net is None:
            path = os.path.join(self.checkpoint_dir, f'bench_{label}.pt')
            net = ThompsonHybridChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[label] = net
        return net

    def _new_game(self):
        sims = (self.fast_sims if self._rng.rand() < self.fast_prob
                else self.full_sims)
        rng = self._rng
        state = self.game.new_initial_state()
        actions = []
        if self._restart_pool and rng.rand() < self.restart_prob:
            seq = self._restart_pool[rng.randint(len(self._restart_pool))]
            pref = restart_prefix(seq, rng, self.restart_k_min,
                                  self.restart_k_max)
            if pref:
                st = self.game.new_initial_state()
                ok = True
                for a in pref:
                    if st.is_terminal() or a not in st.legal_actions():
                        ok = False
                        break
                    st.apply_action(int(a))
                if ok and not st.is_terminal():
                    state, actions = st, list(pref)
        slot = {'state': state, 'hist': [], 'aux': [], 'actions': actions,
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None,
                'root_v0': None, 'best_swing': -1.0, 'best_len': 0}
        if self.pool_prob > 0 and rng.rand() < self.pool_prob:
            try:
                labels = [f[6:-3] for f in os.listdir(self.checkpoint_dir)
                          if f.startswith('bench_') and f.endswith('.pt')] \
                    if self.checkpoint_dir else []
            except OSError:
                labels = []
            if not labels or rng.rand() < self.random_pool_frac:
                slot['pool'] = {'label': 'random',
                                'side': int(rng.randint(0, 2)), 'net': None}
            else:
                label = labels[rng.randint(len(labels))]
                slot['pool'] = {'label': label,
                                'side': int(rng.randint(0, 2)),
                                'net': self._load_pool_net(label)}
        return slot

    def _resolve_pool_moves(self):
        done = []
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            if pool['label'] == 'random':
                legal = state.legal_actions()
                action = int(legal[self._rng.randint(len(legal))])
            else:
                action = policy_action(pool['net'], state, 'cpu', sample=False)
            s['root'] = _descend(s['root'], action)
            state.apply_action(action)
            s['actions'].append(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(self._finish_slot(i))
        return done

    def _finish_slot(self, i):
        s = self.slots[i]
        st = s['state']
        term = st.is_terminal()
        if term:
            ret = st.returns()
            hist2 = resolve_z(s['hist'], ret)
            result = 'draw' if (ret[0] == 0.0 and ret[1] == 0.0) else 'decisive'
            if self.restart_prob > 0 and result == 'decisive':
                self._push_seed(s['actions'])
        else:
            hist2 = strip_episode_meta(s['hist'])
            result = 'cutoff'
        if self.restart_prob > 0 and s['best_swing'] >= self.swing_thresh:
            self._push_seed(s['actions'][:s['best_len']])
        self.last_aux = len(s['aux'])
        self.last_restart_pool = len(self._restart_pool)
        self.stats['games'] += 1
        self.stats[result] += 1
        self.stats['plies'] += int(s['move'])
        # Tag played-path vs solver-aux samples: (…, solved, on_path, z_col).
        # Played carry the game-outcome column z (scored as a separate single-
        # draw likelihood in training); solver-aux carry z=-1 (exact labels).
        played = [(o, vc, li, vis, False, True, z) for (o, vc, li, vis, z) in hist2]
        aux    = [(o, vc, li, vis, True, False, -1) for (o, vc, li, vis) in s['aux']]
        self.slots[i] = self._new_game()
        return played + aux

    def episodes(self):
        while True:
            for data in self._step():
                self.last_lam = self.calib.lam()
                yield data

    def _step(self):
        rng     = self._rng
        done    = self._resolve_pool_moves()
        pending = []            # (slot_idx, path, node, idx)
        jobs    = []            # (kind, i, node, idx, state)
        rows    = []
        seen    = set()
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue
            if s['root'] is None:
                jobs.append(('root', i, None, None, s['state']))
                rows.append(s['state'])
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue
            wave = min(self.wave, s['sims'] - s['n'])
            opening = set()
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], s['state'], rng,
                                              self.calib, opening)
                if st is None:
                    _backup(path); _solver_sweep(path, s['aux'], self.conf_cap)
                    s['n'] += 1
                    continue
                node, idx = edge
                opening.add((id(node), idx))
                pending.append((i, path, node, idx))
                if (id(node), idx) not in seen:
                    seen.add((id(node), idx))
                    jobs.append(('leaf', i, node, idx, st))
                    rows.append(st)

        pre = []
        if rows:
            vp, vc, pl, pc, obs16 = _nn_eval(self.network, self.device, rows)
            lam = self.calib.lam()
            for j, (kind, i, node, idx, st) in enumerate(jobs):
                legal = st.legal_actions()
                child = _node_from_eval(st, vp[j], float(vc[j]),
                                        pl[j][legal], float(pc[j]), lam,
                                        self.calib)
                child.obs = obs16[j]
                if kind == 'root':
                    self.slots[i]['root'] = child
                    self.slots[i]['root_v0'] = _root_value(child)
                else:
                    m0 = float(node.edge[idx] @ _GRID_V)
                    v0 = max(float(node.edge[idx] @ _GRID_V2) - m0 * m0, 0.0)
                    node.children[idx] = child
                    pre.append((node, idx, m0, v0))
        for i, path, node, idx in pending:
            _backup(path)
            self.slots[i]['n'] += 1
        for i, path, node, idx in pending:
            _solver_sweep(path, self.slots[i]['aux'], self.conf_cap)
        for node, idx, m0, v0 in pre:
            _edge_innovation(node, idx, m0, v0, self.calib)

        for i, s in enumerate(self.slots):
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            self._play_move(i)
            if s['state'].is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(self._finish_slot(i))
        return done

    def _play_move(self, i):
        s     = self.slots[i]
        state = s['state']
        root  = s['root']
        obs   = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))
        vc = state_value_target(root, self.conf_cap)
        li, vis = policy_target(root)
        action = root_pick(root, self._rng,
                           thompson=(s['move'] < self.temp_threshold))
        s['hist'].append((obs, vc, li, vis, root.player,
                          _node_solved_outcome(root) is not None))
        s['actions'].append(int(action))
        if self.restart_prob > 0 and s['root_v0'] is not None:
            swing = abs(_root_value(root) - s['root_v0'])
            if swing > s['best_swing']:
                s['best_swing'] = swing
                s['best_len'] = len(s['actions'])
        s['root'] = _descend(root, action)
        s['root_v0'] = (_root_value(s['root']) if s['root'] is not None
                        else None)
        state.apply_action(action)
        s['move'] += 1
        s['n']     = 0

In [ ]:
# ── Multiprocessing self-play + central GPU inference server ─────────────────
# N CPU worker processes do ALL game and tree work (pure Python — processes
# sidestep the GIL) and ship their NN requests to ONE server thread here, which
# batches requests from all workers into single large forward passes on the
# GPU. Each simulation opens one leaf (one eval), so a request is a stack of
# leaf positions + their legal actions; the server gathers the policy logits to
# the legal entries before responding. Small-batch inference (eval, arena) runs
# on CPU replicas elsewhere.
#
# Windows/Jupyter constraint: spawned processes cannot pickle notebook-cell
# functions, so the worker code is written to a real module file first.
import pathlib

_WORKER_SRC = r'''# AUTO-GENERATED by chess_thompson_hybrid_training.ipynb — do not edit by hand.
# Self-play worker module: hybrid state-value + policy-prior Dirichlet-MCTS
# tree ops + worker loop. Loads chess directly (pyspiel.load_game('chess')).
# Exists as a real file because Windows multiprocessing (spawn) cannot pickle
# functions defined inside notebook cells.

# ═══════════════════════════════════════════════════════════════════════════════
# Hybrid Bayesian-MCTS tree ops — standalone copy of the notebook's semantics
# (state-value Dirichlet beliefs + a policy-prior ordering head; ONE eval per
# node expansion; two-stage Thompson selection; MCTS-Solver). Workers are pure
# CPU: no torch import; every NN evaluation is shipped to the main process's
# GPU inference server as a batch of leaf positions + their legal actions (the
# server gathers the policy logits to the legal entries before responding).
# ═══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np
import pyspiel

_GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = _GAME.num_distinct_actions()

_WIN, _DRAW, _LOSS = 0, 1, 2
_FLIP_TERM = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)
ALPHA_FLOOR = 0.05

_GRID_G  = 33
_GRID_X  = (np.arange(_GRID_G) + 0.5) / _GRID_G
_GRID_V  = 2.0 * _GRID_X - 1.0
_GRID_V2 = _GRID_V ** 2
_GRID_LX  = np.log(_GRID_X)
_GRID_L1X = np.log1p(-_GRID_X)
_SPIKE   = np.eye(_GRID_G)
_SPIKE_WIN, _SPIKE_LOSS, _SPIKE_DRAW = (_SPIKE[_GRID_G - 1], _SPIKE[0],
                                        _SPIKE[_GRID_G // 2])
_VLOSS_PEN = 1.0
_MAX_FRAC  = 0.0
_C_PUCT = 1.5            # persistent policy-prior exploration (PUCT U term)
_FPU_REDUCTION = 0.25    # pessimize unvisited moves
_POLICY_CONC_CAP = 40.0  # cap the policy visit-target concentration
_FORCING_MAXLEGAL = 12   # low-branching terminal scan threshold (see tree cell)


_TDOF    = 4.0
_VAR_RES = (2.0 / _GRID_G) ** 2 / 12.0

def _psi(z2):
    return (_TDOF + 1.0) * z2 / (_TDOF + z2)

_zg = np.linspace(-10.0, 10.0, 8001)
_PSI_NORM = float(np.sum(_psi(_zg ** 2)
                         * np.exp(-0.5 * _zg ** 2) / np.sqrt(2.0 * np.pi))
                  * (_zg[1] - _zg[0]))
del _zg


class _Calib:
    __slots__ = ('s', 'n', 'd')

    def __init__(self, prior_n=50.0, halflife=2000.0):
        self.s = prior_n
        self.n = prior_n
        self.d = 0.5 ** (1.0 / halflife)

    def observe(self, e2, var_sum):
        z2 = e2 / (var_sum + 2.0 * _VAR_RES)
        self.s = self.s * self.d + _psi(z2) / _PSI_NORM
        self.n = self.n * self.d + 1.0

    def lam(self):
        return self.s / self.n


def _beta_pmf_rows(alpha, beta):
    logw = ((alpha[:, None] - 1.0) * _GRID_LX[None, :]
            + (beta[:, None] - 1.0) * _GRID_L1X[None, :])
    logw -= logw.max(axis=1, keepdims=True)
    w = np.exp(logw)
    return w / w.sum(axis=1, keepdims=True)


def _flip_pmf(pmf):
    return pmf[::-1].copy()


def _dirichlet_leaf_belief(alpha_w, alpha_d, alpha_l, lam=1.0):
    C = alpha_w + alpha_d + alpha_l
    Ed = alpha_d / C
    Vd = alpha_d * (alpha_w + alpha_l) / (C**2 * (C + 1.0))
    Eq = alpha_w / (alpha_w + alpha_l)
    Vq = alpha_w * alpha_l / ((alpha_w + alpha_l)**2 * (alpha_w + alpha_l + 1.0))
    EX, VX = 1.0 - Ed, Vd
    EY, VY = 2.0 * Eq - 1.0, 4.0 * Vq
    Ev = EX * EY
    Vv = EX**2 * VY + EY**2 * VX + VX * VY
    mu01  = (Ev + 1.0) / 2.0
    var01 = np.clip(Vv / 4.0 * lam, 1e-9, None)
    var01 = np.minimum(var01, mu01 * (1.0 - mu01) * 0.999)
    conc  = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    a_beta = np.maximum(mu01 * conc, ALPHA_FLOOR)
    b_beta = np.maximum((1.0 - mu01) * conc, ALPHA_FLOOR)
    return _beta_pmf_rows(a_beta, b_beta), Ed


def _prob_best(E, C):
    logG = np.log(np.clip(C - 0.5 * E, 1e-12, 1.0))
    loo  = np.exp(logG.sum(axis=0)[None, :] - logG)
    w = (E * loo).sum(axis=1)
    s = w.sum()
    return w / s if s > 0 else np.full(len(w), 1.0 / len(w))


class _TNode:
    __slots__ = ('player', 'legal', 'vself', 'dself', 'edge', 'draw', 'pol',
                 'beta', 'visits', 'vloss', 'term', 'children', 'obs')

    def __init__(self, player, legal, v_probs, v_conf, pol, beta, lam=1.0):
        self.player = player
        self.legal  = np.asarray(legal, dtype=np.int32)
        self.obs    = None
        k = len(self.legal)
        vp = np.asarray(v_probs, dtype=np.float64)
        cf = float(v_conf)
        aw = max(cf * vp[0], ALPHA_FLOOR)
        ad = max(cf * vp[1], ALPHA_FLOOR)
        al = max(cf * max(vp[2], 0.0), ALPHA_FLOOR)
        vpmf, dd = _dirichlet_leaf_belief(np.array([aw]), np.array([ad]),
                                          np.array([al]), lam)
        self.vself = vpmf[0]
        self.dself = float(dd[0])
        self.edge  = np.tile(self.vself, (k, 1))
        self.draw  = np.full(k, self.dself)
        self.pol   = np.asarray(pol, dtype=np.float64)
        self.beta  = float(beta)
        self.visits   = np.zeros(k, dtype=np.int64)
        self.vloss    = np.zeros(k, dtype=np.int32)
        self.term     = np.full(k, -1, dtype=np.int8)
        self.children = [None] * k


_TERM_PMF  = np.stack([_SPIKE_WIN, _SPIKE_DRAW, _SPIKE_LOSS])
_TERM_DRAW_VAL = np.array([0.0, 1.0, 0.0])


def _set_term(node, idx, outcome):
    node.term[idx] = outcome
    node.edge[idx] = _TERM_PMF[outcome]
    node.draw[idx] = _TERM_DRAW_VAL[outcome]


def _opened_mask(node):
    m = (node.term >= 0)
    for i, c in enumerate(node.children):
        if c is not None:
            m[i] = True
    return m


def _node_beliefs(node):
    mask = _opened_mask(node)
    if not mask.any():
        return node.vself.copy(), node.dself
    E = node.edge[mask]
    D = node.draw[mask]
    if E.shape[0] == 1:
        return E[0].copy(), float(D[0])
    C = np.cumsum(E, axis=1)
    np.clip(C, 0.0, 1.0, out=C)
    w = _prob_best(E, C)
    mix = w @ E
    mean_draw = float(w @ D)
    if _MAX_FRAC > 0.0:
        Fmax = C.prod(axis=0)
        mx = np.empty(_GRID_G)
        mx[0]  = Fmax[0]
        mx[1:] = np.diff(Fmax)
        np.maximum(mx, 0.0, out=mx)
        ms = mx.sum()
        mix = (1.0 - _MAX_FRAC) * mix + _MAX_FRAC * (mx / ms if ms > 0 else mix)
    s = mix.sum()
    return (mix / s if s > 0 else np.full(_GRID_G, 1.0 / _GRID_G)), mean_draw


def _sample_pmf(pmf, u):
    c = np.cumsum(pmf)
    idx = min(int((c < u).sum()), _GRID_G - 1)
    return _GRID_V[idx]


def _choose_edge(node, rng, exclude=None):
    # PUCT-style Thompson selection: Thompson-sample every edge's value belief
    # (unopened = FPU prior, pessimized; proven = spike), add a persistent
    # policy-prior bonus U = c_puct·P(a)·sqrt(ΣN)/(1+N(a)) that decays with
    # visits, argmax. See the tree-ops cell for the rationale.
    C = np.cumsum(node.edge, axis=1)
    u = rng.random_sample(len(node.legal))
    gi = np.minimum((C < u[:, None]).sum(axis=1), _GRID_G - 1)
    v = _GRID_V[gi].copy()
    opened = _opened_mask(node)
    if not opened.all():
        v[~opened] -= _FPU_REDUCTION
    N = node.visits.astype(np.float64)
    U = _C_PUCT * node.pol * np.sqrt(N.sum() + 1.0) / (1.0 + N)
    score = v + U - _VLOSS_PEN * node.vloss
    if exclude:
        masked = score.copy()
        for i in exclude:
            if 0 <= i < len(masked):
                masked[i] = -np.inf
        if np.isfinite(masked).any():
            score = masked
    return int(score.argmax())


def _select_leaf(root, root_state, rng, calib=None, exclude=None):
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        ex = None
        if exclude:
            ex = {i for (nid, i) in exclude if nid == id(node)}
        idx = _choose_edge(node, rng, ex)
        node.vloss[idx] += 1
        node.visits[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        child = node.children[idx]
        if child is None:
            state.apply_action(int(node.legal[idx]))
            if state.is_terminal():
                r = state.returns()[node.player]
                if calib is not None:
                    m0 = float(node.edge[idx] @ _GRID_V)
                    v0 = max(float(node.edge[idx] @ _GRID_V2) - m0 * m0, 0.0)
                    calib.observe((m0 - r) ** 2, v0)
                _set_term(node, idx,
                          _WIN if r > 0 else (_LOSS if r < 0 else _DRAW))
                return path, None, None
            return path, state, (node, idx)
        state.apply_action(int(node.legal[idx]))
        node = child


def _backup(path):
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        if node.term[idx] < 0 and node.children[idx] is not None:
            vpmf, mdraw = _node_beliefs(node.children[idx])
            node.edge[idx] = _flip_pmf(vpmf)
            node.draw[idx] = mdraw


def _edge_innovation(node, idx, m0, v0, calib):
    p = node.edge[idx]
    m1 = float(p @ _GRID_V)
    v1 = max(float(p @ _GRID_V2) - m1 * m1, 0.0)
    calib.observe((m0 - m1) ** 2, v0 + v1)


def _node_solved_outcome(node):
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _solver_sweep(path, aux=None, conf_cap=None):
    for k in range(len(path) - 1, -1, -1):
        parent, idx = path[k]
        child = parent.children[idx]
        if child is None:
            break
        out = _node_solved_outcome(child)
        if out is None or parent.term[idx] >= 0:
            break
        _set_term(parent, idx, int(_FLIP_TERM[out]))
        if aux is not None and child.obs is not None:
            aux.append((child.obs, state_value_target(child, conf_cap))
                       + policy_target(child))


def state_value_target(node, conf_cap):
    out = _node_solved_outcome(node)
    if out is not None:
        cnt = np.full(3, ALPHA_FLOOR, dtype=np.float32)
        cnt[out] = conf_cap - 2.0 * ALPHA_FLOOR
        return cnt
    vpmf, d = _node_beliefs(node)
    mean_v = float(vpmf @ _GRID_V)
    var_v  = max(float(vpmf @ _GRID_V2) - mean_v * mean_v, 1e-9)
    mu01   = (mean_v + 1.0) / 2.0
    var01  = max(var_v / 4.0, 1e-9)
    conc   = max(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    conc   = min(conc, conf_cap)
    nondraw = min(max(1.0 - d, 0.0), 1.0)
    p_win  = min(max((mean_v + nondraw) / 2.0, 0.0), nondraw)
    p_loss = nondraw - p_win
    return (np.array([p_win, d, p_loss]) * conc).astype(np.float32)


def policy_target(node):
    out = _node_solved_outcome(node)
    if out is not None:
        vc = np.full(len(node.legal), 0.0, dtype=np.float32)
        vc[_solved_best_edge(node, out)] = _POLICY_CONC_CAP
        return node.legal.astype(np.int32), vc
    vis = node.visits.astype(np.float64)
    tot = vis.sum()
    if tot > _POLICY_CONC_CAP:
        vis = vis * (_POLICY_CONC_CAP / tot)
    return node.legal.astype(np.int32), vis.astype(np.float32)


def _solved_best_edge(node, out):
    if out == _WIN:
        return int(np.nonzero(node.term == _WIN)[0][0])
    if out == _DRAW:
        return int(np.nonzero(node.term == _DRAW)[0][0])
    return int(node.visits.argmax())


def _descend(root, action):
    if root is None:
        return None
    hit = np.nonzero(root.legal == action)[0]
    return root.children[int(hit[0])] if len(hit) else None


def resolve_z(samples, returns):
    # Attach the game-outcome column z_col (0/1/2, mover's perspective) to each
    # on-path sample WITHOUT blending it into the value counts — z is scored as
    # a separate single-draw likelihood in training so outcome variance bounds
    # confidence with no cap. Proven played edges get z=-1. See the tree-ops
    # cell's resolve_z for the full rationale.
    out = []
    for obs, vc, li, vis, player, proven in samples:
        if proven:
            z = -1
        else:
            r = returns[player]
            z = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
        out.append((obs, vc, li, vis, z))
    return out


def strip_episode_meta(samples):
    return [(o, vc, li, vis, -1) for o, vc, li, vis, _p, _v in samples]


def restart_prefix(seq, rng, k_min, k_max):
    if len(seq) <= k_min:
        return None
    k = int(rng.randint(k_min, min(k_max, len(seq) - 1) + 1))
    return seq[:len(seq) - k]


def _root_value(node):
    vpmf, _ = _node_beliefs(node)
    return float(vpmf @ _GRID_V)


def _softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


def _scan_terminal_children(node, state, calib=None):
    # Low-branching-node solver signal: mark terminal children proven (no NN),
    # feeding ground-truth innovations to calib. See the tree-ops cell.
    me = node.player
    for i, a in enumerate(node.legal):
        ch = state.clone()
        ch.apply_action(int(a))
        if ch.is_terminal():
            r = ch.returns()[me]
            if calib is not None:
                m0 = float(node.edge[i] @ _GRID_V)
                v0 = max(float(node.edge[i] @ _GRID_V2) - m0 * m0, 0.0)
                calib.observe((m0 - r) ** 2, v0)
            _set_term(node, i, _WIN if r > 0 else (_LOSS if r < 0 else _DRAW))


def _node_from_eval(state, v_probs, v_conf, p_logits_legal, p_conf, lam,
                    calib=None):
    legal = state.legal_actions()
    pol   = _softmax(np.asarray(p_logits_legal, dtype=np.float64))
    node  = _TNode(state.current_player(), legal, v_probs, float(v_conf),
                   pol, float(p_conf), lam)
    node.obs = np.asarray(state.observation_tensor(state.current_player()),
                          dtype=np.float16)
    if len(legal) <= _FORCING_MAXLEGAL:
        _scan_terminal_children(node, state, calib)
    return node


def root_pick(root, rng, thompson):
    # Commit a move from the search statistics. VISIT-BASED (AlphaZero) rather
    # than value-mean-based: visit counts integrate the value posterior over
    # every simulation, so they are robust to a single lucky Thompson draw on a
    # barely-visited edge — the failure mode that made a weak value head commit
    # rim/rook moves. The Thompson posterior still drives the SEARCH itself
    # (edge selection); this is only the final commit. See the tree-ops cell.
    opened = _opened_mask(root)
    if not opened.any():
        return int(root.legal[int(root.pol.argmax())])
    if thompson:                                   # temperature: sample ∝ visits
        w = root.visits.astype(np.float64)
        s = w.sum()
        if s <= 0:
            return int(root.legal[int(root.pol.argmax())])
        i = int((np.cumsum(w / s) < rng.random_sample()).sum())
        return int(root.legal[min(i, len(root.legal) - 1)])
    wins = np.nonzero(root.term == _WIN)[0]        # greedy: proven win first
    pool = wins if len(wins) else np.nonzero(opened)[0]
    vis = root.visits[pool]
    cand = pool[vis == vis.max()]                  # most-visited; ties by value
    if len(cand) > 1:
        vmean = root.edge[cand] @ _GRID_V
        cand = cand[vmean >= vmean.max() - 1e-9]
        i = int(cand[rng.randint(len(cand))])
    else:
        i = int(cand[0])
    return int(root.legal[i])


def policy_action_logits(p_logits_legal, p_conf, legal, rng, sample):
    p = _softmax(np.asarray(p_logits_legal, dtype=np.float64))
    if sample:
        g = rng.standard_gamma(np.maximum(float(p_conf) * p, 1e-9))
        return int(legal[int(g.argmax())])
    return int(legal[int(p.argmax())])


def run_worker(worker_id, req_q, resp_q, pool_resp_q, episode_q, cfg):
    """Pipelined self-play worker (leapfrog halves). NN requests are tagged
    'live' (the training net) or a benchmark label (a frozen opponent). Wire
    format: a request is `(worker_id, net_id, obs (n,1280) fp16,
    legals [int32])` — one row per LEAF POSITION to open (plus its legal
    actions, which the server needs to gather the policy logits). The response
    is `[(v3 (3,), vconf, p_legal (m,), pconf), ...]` — the value head raw
    (4 floats) + the policy logits gathered to the m legal moves (~35 floats),
    so the response queue ships ~0.15 KB/leaf instead of the dense 18.7 KB
    policy row."""
    rng  = np.random.RandomState(cfg['seed'] + worker_id * 7919)
    game = _GAME
    conf_cap  = cfg['conf_cap']
    pool_prob = cfg.get('pool_prob', 0.0)
    pool_dir  = cfg.get('checkpoint_dir')
    max_plies = cfg.get('max_selfplay_plies', 400)
    z_mix     = cfg.get('z_mix', 0.5)
    z_gamma   = cfg.get('z_gamma', 1.0)
    restart_prob  = cfg.get('restart_prob', 0.0)
    restart_kmin  = cfg.get('restart_k_min', 2)
    restart_kmax  = cfg.get('restart_k_max', 30)
    restart_cap   = cfg.get('restart_pool_cap', 128)
    swing_thresh  = cfg.get('swing_thresh', 0.6)
    rand_pool_frac = cfg.get('random_pool_frac', 0.5)
    restart_pool = []
    calib = _Calib()

    def _push_seed(seq):
        if len(seq) >= 2:
            restart_pool.append(list(seq))
            if len(restart_pool) > restart_cap:
                del restart_pool[0]

    def new_game():
        sims = (cfg['fast_sims'] if rng.rand() < cfg['fast_prob']
                else cfg['full_sims'])
        state = game.new_initial_state()
        actions = []
        if restart_pool and rng.rand() < restart_prob:
            seq = restart_pool[rng.randint(len(restart_pool))]
            pref = restart_prefix(seq, rng, restart_kmin, restart_kmax)
            if pref:
                st = game.new_initial_state()
                ok = True
                for a in pref:
                    if st.is_terminal() or a not in st.legal_actions():
                        ok = False
                        break
                    st.apply_action(int(a))
                if ok and not st.is_terminal():
                    state, actions = st, list(pref)
        slot = {'state': state, 'hist': [], 'aux': [], 'actions': actions,
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None,
                'root_v0': None, 'best_swing': -1.0, 'best_len': 0}
        if pool_prob > 0 and rng.rand() < pool_prob:
            try:
                labels = [f[6:-3] for f in os.listdir(pool_dir)
                          if f.startswith('bench_') and f.endswith('.pt')] \
                    if pool_dir else []
            except OSError:
                labels = []
            if not labels or rng.rand() < rand_pool_frac:
                label = 'random'
            else:
                label = labels[rng.randint(len(labels))]
            slot['pool'] = {'label': label, 'side': int(rng.randint(0, 2))}
        return slot

    def finish_and_reset(i):
        s = slots[i]
        st = s['state']
        term = st.is_terminal()
        if term:
            ret = st.returns()
            hist2 = resolve_z(s['hist'], ret)
            result = 'draw' if (ret[0] == 0.0 and ret[1] == 0.0) else 'decisive'
            if restart_prob > 0 and result == 'decisive':
                _push_seed(s['actions'])
        else:
            hist2 = strip_episode_meta(s['hist'])
            result = 'cutoff'
        if restart_prob > 0 and s['best_swing'] >= swing_thresh:
            _push_seed(s['actions'][:s['best_len']])
        # Tag played-path vs solver-aux: (…, solved, on_path, z_col).
        played = [(o, vc, li, vis, False, True, z) for (o, vc, li, vis, z) in hist2]
        aux    = [(o, vc, li, vis, True, False, -1)
                  for (o, vc, li, vis) in s['aux']]
        episode_q.put((played + aux, calib.lam(), len(s['aux']),
                       result, int(s['move']), len(restart_pool)))
        slots[i] = new_game()

    slots = [new_game() for _ in range(cfg['games_per_worker'])]
    mid = max(1, cfg['games_per_worker'] // 2)
    halves = [list(range(mid)), list(range(mid, cfg['games_per_worker']))]

    def collect(idxs):
        """Root expansions + one leaf-open wave per game. Proven edges backed
        up immediately; pool turns + solved roots skipped. Returns (jobs,
        paths, obs, legals): jobs are the unique leaf/root evals, paths every
        pending selection."""
        jobs, paths, obs, legals = [], [], [], []
        seen = set()
        for i in idxs:
            s = slots[i]
            st0 = s['state']
            pool = s['pool']
            if pool is not None and st0.current_player() == pool['side']:
                continue
            if s['root'] is None:
                o = np.asarray(st0.observation_tensor(st0.current_player()),
                               dtype=np.float16)
                leg = st0.legal_actions()
                jobs.append(('root', i, None, None, st0, leg))
                obs.append(o); legals.append(np.asarray(leg, dtype=np.int32))
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue
            wave = min(cfg['wave'], s['sims'] - s['n'])
            opening = set()
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], st0, rng, calib,
                                              opening)
                if st is None:
                    _backup(path); _solver_sweep(path, s['aux'], conf_cap)
                    s['n'] += 1
                    continue
                node, idx = edge
                opening.add((id(node), idx))
                paths.append((i, path, node, idx))
                if (id(node), idx) not in seen:
                    seen.add((id(node), idx))
                    leg = st.legal_actions()
                    o = np.asarray(st.observation_tensor(st.current_player()),
                                   dtype=np.float16)
                    jobs.append(('leaf', i, node, idx, st, leg))
                    obs.append(o); legals.append(np.asarray(leg, dtype=np.int32))
        return jobs, paths, obs, legals

    def apply_and_advance(idxs, jobs, paths, resp):
        lam = calib.lam()
        pre = {}
        if resp is not None:
            for e, (v3, vconf, p_leg, pconf) in zip(jobs, resp):
                kind, i, node, idx, st, leg = e
                child = _node_from_eval(st, v3, vconf, p_leg, pconf, lam, calib)
                if kind == 'root':
                    slots[i]['root'] = child
                    slots[i]['root_v0'] = _root_value(child)
                else:
                    m0 = float(node.edge[idx] @ _GRID_V)
                    v0 = max(float(node.edge[idx] @ _GRID_V2) - m0 * m0, 0.0)
                    node.children[idx] = child
                    pre[(id(node), idx)] = (node, idx, m0, v0)
        for i, path, node, idx in paths:
            _backup(path)
            slots[i]['n'] += 1
        for i, path, node, idx in paths:
            _solver_sweep(path, slots[i]['aux'], conf_cap)
        for node, idx, m0, v0 in pre.values():
            _edge_innovation(node, idx, m0, v0, calib)
        for i in idxs:
            s = slots[i]
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            state = s['state']
            root = s['root']
            o = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))
            vc = state_value_target(root, conf_cap)
            li, vis = policy_target(root)
            action = root_pick(root, rng,
                               thompson=(s['move'] < cfg['temp_threshold']))
            s['hist'].append((o, vc, li, vis, root.player,
                              _node_solved_outcome(root) is not None))
            s['actions'].append(int(action))
            if restart_prob > 0 and s['root_v0'] is not None:
                swing = abs(_root_value(root) - s['root_v0'])
                if swing > s['best_swing']:
                    s['best_swing'] = swing
                    s['best_len'] = len(s['actions'])
            s['root'] = _descend(root, action)
            s['root_v0'] = (_root_value(s['root']) if s['root'] is not None
                            else None)
            state.apply_action(action)
            s['move'] += 1
            s['n'] = 0
            if state.is_terminal() or s['move'] >= max_plies:
                finish_and_reset(i)

    def resolve_pool_moves(idxs):
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            legal = state.legal_actions()
            if pool['label'] == 'random':
                action = int(legal[rng.randint(len(legal))])
            else:
                obs = np.asarray(
                    state.observation_tensor(state.current_player()),
                    dtype=np.float16)
                req_q.put((worker_id, pool['label'], obs[None],
                           [np.asarray(legal, dtype=np.int32)]))
                (v3, vconf, p_leg, pconf), = pool_resp_q.get()
                action = policy_action_logits(p_leg, pconf, legal, rng, False)
            s['root'] = _descend(s['root'], action)
            state.apply_action(action)
            s['actions'].append(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= max_plies:
                finish_and_reset(i)

    inflight = [None, None]
    while True:
        for h in (0, 1):
            if inflight[h] is not None:
                jobs, paths, sent = inflight[h]
                resp = resp_q.get() if sent else None
                apply_and_advance(halves[h], jobs, paths, resp)
                inflight[h] = None
            resolve_pool_moves(halves[h])
            jobs, paths, obs, legals = collect(halves[h])
            sent = False
            if obs:
                req_q.put((worker_id, 'live', np.stack(obs), legals))
                sent = True
            inflight[h] = (jobs, paths, sent)
'''

pathlib.Path('chess_thompson_hybrid_worker.py').write_text(_WORKER_SRC, encoding='utf-8')

import multiprocessing as _mp
import threading
import queue as _queue
import time as _time
import os as _os
import sys as _sys


class MPSelfPlayPool:
    """n_workers CPU processes play games; an inference-server thread in this
    process batches ALL their NN requests into single forward passes on
    `device`. Exposes the same .episodes() generator as ThompsonParallelSelfPlay.
    `lock` serializes model access between the server thread and training.

    Wire format: a request is a stack of leaf positions + their legal actions.
    The VALUE head (n,3)+(n,) is downloaded whole (16 bytes/row); only the
    POLICY logits (dense (n,4674)) are gathered to the ~35 legal entries per
    row — on-device index_select where the probe supports it, so ~1% of the
    policy head ever crosses the bus."""

    def __init__(self, network, device, n_workers, cfg, batch_window_s=0.002,
                 checkpoint_dir=None, channels=None, num_blocks=None):
        self.network = network
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU net
        self.lock = threading.Lock()
        self._stop = threading.Event()
        ctx = _mp.get_context('spawn')            # Windows-compatible
        self.req_q = ctx.Queue()
        self.episode_q = ctx.Queue(maxsize=64)    # backpressure on workers
        self.resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.pool_resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.window = batch_window_s
        # Initialize the autograd engine's per-device state from the MAIN
        # thread before any other thread touches the device (torch-directml
        # "device_ready_queues_" assert otherwise).
        if str(device) != 'cpu':
            _t = torch.zeros(4, device=device, requires_grad=True)
            (_t * 2.0).sum().backward()
        if _os.getcwd() not in _sys.path:
            _sys.path.insert(0, _os.getcwd())
        import chess_thompson_hybrid_worker
        self.procs = [ctx.Process(target=chess_thompson_hybrid_worker.run_worker,
                                  args=(i, self.req_q, self.resp_qs[i],
                                        self.pool_resp_qs[i],
                                        self.episode_q, cfg), daemon=True)
                      for i in range(n_workers)]
        for p in self.procs:
            p.start()
        self.server = threading.Thread(target=self._serve, daemon=True)
        self.server.start()

    def _get_net(self, net_id):
        if net_id == 'live':
            return self.network, self.device, True
        net = self._pool_nets.get(net_id)
        if net is None:
            path = _os.path.join(self.checkpoint_dir, f'bench_{net_id}.pt')
            net = ThompsonHybridChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[net_id] = net
        return net, 'cpu', False

    def _forward(self, net, net_device, xin, flat):
        """One forward pass. Returns (v_probs (N,3), v_conf (N,),
        p_logits_gathered (M,), p_conf (N,)) as numpy — the value + confidence
        heads whole, the policy logits gathered at `flat` (int64 indices into
        the flattened (N·A) policy axis) on-device where supported."""
        x = torch.from_numpy(xin).to(net_device)
        vp, vc, pl, pc = net(x)
        vp_np = vp.cpu().numpy(); vc_np = vc.cpu().numpy(); pc_np = pc.cpu().numpy()
        if _GATHER_FWD_OK and str(net_device) != 'cpu':
            ft = torch.from_numpy(flat).to(net_device)
            pg = pl.reshape(-1).index_select(0, ft).cpu().numpy()
        else:
            pg = pl.reshape(-1).cpu().numpy()[flat]
        return vp_np, vc_np, pg, pc_np

    def _serve(self):
        A = _NUM_ACTIONS
        while not self._stop.is_set():
            try:
                reqs = [self.req_q.get(timeout=0.1)]
            except _queue.Empty:
                continue
            rows = reqs[0][2].shape[0]
            deadline = _time.monotonic() + self.window
            while _time.monotonic() < deadline and rows < 2048:
                try:
                    r = self.req_q.get_nowait()
                    reqs.append(r)
                    rows += r[2].shape[0]
                except _queue.Empty:
                    _time.sleep(0.0003)
            groups = {}
            for wid, net_id, obs, legals in reqs:
                groups.setdefault(net_id, []).append((wid, obs, legals))
            for net_id, group in groups.items():
                obs = np.concatenate([o for _, o, _ in group], axis=0)
                xin = obs.reshape(-1, *_OBS_SHAPE).astype(np.float32)
                row_legals = [l for _, _, ls in group for l in ls]
                flat = np.concatenate([l.astype(np.int64) + r * A
                                       for r, l in enumerate(row_legals)]) \
                    if row_legals else np.zeros(0, np.int64)
                offs = np.zeros(len(row_legals) + 1, dtype=np.int64)
                np.cumsum([len(l) for l in row_legals], out=offs[1:])
                # A failed batch must NOT kill the server (every waiting worker
                # would deadlock) — answer with neutral priors and keep serving.
                try:
                    net, net_device, needs_lock = self._get_net(net_id)
                    if needs_lock:
                        with self.lock, torch.no_grad():
                            vp, vc, pg, pc = self._forward(net, net_device,
                                                           xin, flat)
                    else:
                        with torch.no_grad():
                            vp, vc, pg, pc = self._forward(net, net_device,
                                                           xin, flat)
                except Exception:
                    import traceback
                    traceback.print_exc()
                    print(f"inference server: batch for '{net_id}' failed — "
                          f"neutral priors", flush=True)
                    n = xin.shape[0]
                    vp = np.full((n, 3), 1.0 / 3.0, np.float32)
                    vc = np.full(n, CONF_MIN, np.float32)
                    pg = np.zeros(len(flat), np.float32)
                    pc = np.full(n, POL_CONF_MIN, np.float32)
                target_qs = self.resp_qs if net_id == 'live' else self.pool_resp_qs
                ri = 0        # running index over rows (aligns v/pc per row)
                for wid, o, ls in group:
                    out = []
                    for _ in ls:
                        a, b = offs[ri], offs[ri + 1]
                        out.append((vp[ri], float(vc[ri]), pg[a:b],
                                    float(pc[ri])))
                        ri += 1
                    target_qs[wid].put(out)

    def episodes(self):
        self.last_lam = 1.0
        self.last_aux = 0
        self.last_restart_pool = 0
        self.stats = {'games': 0, 'draw': 0, 'cutoff': 0, 'decisive': 0,
                      'plies': 0}
        while True:
            samples, lam, n_aux, result, plies, rpool = self.episode_q.get()
            self.last_lam = lam
            self.last_aux = n_aux
            self.last_restart_pool = rpool
            self.stats['games'] += 1
            self.stats[result] = self.stats.get(result, 0) + 1
            self.stats['plies'] += plies
            yield samples

    def shutdown(self):
        self._stop.set()
        try:
            self.server.join(timeout=2.0)
        except Exception:
            pass
        for p in self.procs:
            p.terminate()
        for p in self.procs:
            p.join(timeout=2.0)
        for q in ([self.req_q, self.episode_q] + self.resp_qs
                  + self.pool_resp_qs):
            try:
                q.close(); q.cancel_join_thread()
            except Exception:
                pass

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# The hybrid restores AlphaZero-style DEPTH: one simulation opens ONE node
# (one NN eval), so the sim budget again buys that many expanded tree nodes
# (thompson_value's child-batch expansion evaluated ~35 children to open a
# single node, capping depth). Budgets are between the two siblings: bigger
# than thompson_value's (each sim is ~7× cheaper — one leaf, not ~35) but the
# policy prior focuses them, so they reach deep forcing lines a flat search
# would miss.
NUM_EPISODES     = 50_000
FULL_MCTS_SIMS   = 400     # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 100     # fast self-play (75%)
FAST_GAME_PROB   = 0.75
MAX_SELFPLAY_PLIES = 400
# ── Hardware parallelism (AMD GPU via DirectML) ──────────────────────────────
# Each simulation is one leaf eval, so a wave is n_games/2 × wave rows — back
# to the thompson_z regime (more concurrent games per worker to fill the GPU
# than thompson_value needed, since expansions no longer ship ~35 rows each).
import os as _os_hw
USE_WORKERS      = True
SELFPLAY_WORKERS = min(8, max(2, (_os_hw.cpu_count() or 8) - 2))
GAMES_PER_WORKER = 32      # per worker, split into two pipelined halves of 16
WORKER_WAVE      = 8       # leaf-opens/game/wave; server batches all workers'
N_PARALLEL_GAMES = 8       # (single-process fallback when USE_WORKERS=False)
WAVE_PER_GAME    = 8
TEMP_THRESHOLD   = 20
CONF_CAP         = 30.0    # value posterior concentration cap. LOWERED from 100:
                           # the first hybrid run pinned predicted α₀ near 95
                           # (confidence runaway) — a tight, wrong value head
                           # makes the search barely explore. A lower cap keeps
                           # value beliefs wide enough for Thompson selection to
                           # actually resolve moves (calibration λ still runs on
                           # top). See the intro's failure-mode note.
# ── The game outcome z: a SEPARATE single-draw likelihood (not blended) ──────
# The earlier design averaged z into the value count vector. That laundered
# away the fact that z is ONE noisy draw: the evidence loss then read an
# extreme blended mean as high-concentration truth, and predicted α₀ ran away
# to the cap. Instead z is now scored as a standalone single-draw likelihood —
# which for a Dirichlet is exactly the cross-entropy of the value MEAN against
# the outcome (P(z)=p_z, independent of concentration). So z densely trains the
# value MEAN (every position toward its game outcome, AlphaZero-style) while the
# value COUNTS (pure search posterior) train the CONCENTRATION honestly, and a
# net that stays overconfident is penalized every time an outcome surprises it —
# outcome variance bounds α₀ with no cap. See resolve_z in cell 5.
Z_LOSS_WEIGHT    = 1.0     # weight on the single-draw z cross-entropy
# ── Prioritized replay ───────────────────────────────────────────────────────
# Played-path positions (on the actually-played trajectory, carrying z) and
# solver-proven positions (exact ground truth) are the only classes emitted;
# both are high quality, so they default to equal weight — the "prioritization"
# is that no low-quality off-path interior nodes dilute the buffer.
PRIORITY_PLAYED  = 1.0
PRIORITY_SOLVER  = 1.0
VALUE_LOSS_WEIGHT  = 1.0   # value head: Dirichlet-Multinomial over (win,draw,loss)
POLICY_LOSS_WEIGHT = 0.25  # policy head: Dirichlet-Categorical over legal moves.
                           # LOWERED from 1.0: a 35-category NLL is intrinsically
                           # ~8× a 3-category one, so at equal weight the policy
                           # loss (~118) dwarfed the value loss (~15) and starved
                           # the value head of gradient. The value head is the
                           # scarce resource, so it gets the larger share.
# ── Backward-restart curriculum (see cell 5) ─────────────────────────────────
RESTART_PROB     = 0.30
RESTART_K_MIN    = 2
RESTART_K_MAX    = 30
RESTART_POOL_CAP = 128
SWING_THRESH     = 0.6
RANDOM_POOL_FRAC = 0.5
SELFPLAY_POOL_PROB = 0.15
EVAL_EVERY       = 50
DEEP_EVAL_EVERY  = 1000     # RAISED from 500: the unified pool plays ~(6+5)·N
                           # games/eval (N = #players, grows 3 per generation),
                           # and the @32/@128 search games are the slow part —
                           # so evals get expensive as the pool grows. Raise
                           # further, or prune old benchmarks, if it dominates.
QUICK_EVAL_GAMES = 20
# ── Unified eval pool (one shared Elo table over checkpoint × sim-budget) ─────
# Every checkpoint enters the pool as ONE player PER sim budget below, plus a
# single 'random' mover. sims=0 is the search-free policy-head argmax; the
# others run the full hybrid search — all rated in ONE Elo table so
# search-vs-no-search and generation-vs-generation are directly comparable.
# When a checkpoint is added, its new players each play EVAL_GAMES_PER_PAIR
# games (alternating colours) vs every existing player; then EVAL_RANDOM_MULT·N
# random-pair games run over all players.
EVAL_SIMS        = [0, 32, 128]
EVAL_GAMES_PER_PAIR = 2     # new-vs-existing games (one White, one Black)
EVAL_RANDOM_MULT = 5        # random-pair refresh games = this × (#players)
EVAL_OPENING_PLIES = 4
START_ELO        = 1000.0
K_FACTOR         = 32.0
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 4      # LOWERED from 8: at 8 each position was reused ~30×,
                           # letting the value head memorize the mean and drive
                           # confidence to the cap. Fewer steps → lower replay
                           # ratio (and faster wall-clock per episode).
MAX_BUFFER       = 100_000
CHANNELS         = 64      # small value/trunk; the policy head (flat, gathered)
NUM_BLOCKS       = 6       # is the bulk of the ~2.9M params — still ~4× smaller
                           # than thompson_z's 12.7M per-action net.
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100
LR_DECAY_EPS     = 30_000
WEIGHT_DECAY     = 1e-4
LR_MIN_FACTOR    = 0.10
CHECKPOINT_DIR   = 'chess_checkpoints_thompson_hybrid'
RESUME_FROM_CHECKPOINT = True

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = GAME
base_network = ThompsonHybridChessNet(CHANNELS, NUM_BLOCKS).to(DEVICE)
network      = base_network

network = base_network
if _BACKEND != 'directml' and hasattr(torch, 'compile'):
    import shutil, platform, logging
    _cc = (shutil.which('cl') if platform.system() == 'Windows'
           else shutil.which('g++') or shutil.which('gcc') or shutil.which('clang'))
    if _cc is None:
        print('torch.compile: skipped (no C++ compiler for Inductor) — eager mode')
    else:
        logging.getLogger('torch.fx.experimental.symbolic_shapes').setLevel(logging.ERROR)
        try:
            _compiled = torch.compile(base_network, dynamic=True)
            with torch.no_grad():
                _compiled(torch.zeros(1, *_OBS_SHAPE, device=DEVICE))
            network = _compiled
            print('torch.compile: enabled')
        except Exception as _e:
            print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')

class LerpFreeAdamW(torch.optim.Optimizer):
    """AdamW from mul_/add_/addcmul_/addcdiv_ only — DirectML lacks aten::lerp.
    Identical math to torch.optim.AdamW."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                st['step'] += 1
                t = int(st['step'])
                m, v = st['exp_avg'], st['exp_avg_sq']
                p.mul_(1.0 - lr * wd)
                m.mul_(b1).add_(p.grad, alpha=1.0 - b1)
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1.0 - b2)
                bc1 = 1.0 - b1 ** t
                bc2 = 1.0 - b2 ** t
                denom = (v.sqrt() / (bc2 ** 0.5)).add_(eps)
                p.addcdiv_(m, denom, value=-(lr / bc1))
        return loss


if _BACKEND == 'directml':
    optimizer = LerpFreeAdamW(network.parameters(),
                              lr=LR_PEAK, weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(network.parameters(),
                                  lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = min(1.0, (ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1))
    return float(max(LR_MIN_FACTOR, 0.5 * (1.0 + np.cos(np.pi * frac))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)


# ── Losses ──────────────────────────────────────────────────────────────────────
# VALUE head: the same Dirichlet-Multinomial evidence NLL / KL as the sibling
# notebooks, now over the single (win,draw,loss) state target. POLICY head: the
# SAME Dirichlet-Multinomial NLL but over the m LEGAL moves — the visit counts
# are a multinomial observation of "which move is best" under the predicted
# Dirichlet-Categorical prior Dir(β·softmax(legal logits)). Both are
# non-self-referential (the prior is scored against search evidence, never
# matched to a target containing itself), so confidence/concentration is
# earned. The policy loss is computed on the GATHERED legal entries (segment
# ops over ~35 of 4674 per row), reusing the on-device index_select + split-
# backward machinery.

def _value_nll(p, c, t):
    """Per-sample negative Dirichlet-Multinomial log-likelihood of counts `t`
    (B,3) under Dirichlet(c·p). Returns (B,)."""
    A = (c.unsqueeze(-1) * p).clamp_min(ALPHA_FLOOR)
    log_ev = ((torch.lgamma(A + t).sum(-1) - torch.lgamma((A + t).sum(-1)))
              - (torch.lgamma(A).sum(-1) - torch.lgamma(A.sum(-1))))
    return -log_ev


def _value_kl(p, c, t):
    """Per-sample KL( Dirichlet(t) ‖ Dirichlet(c·p) ). Returns (B,)."""
    tt = t.clamp_min(ALPHA_FLOOR)
    b  = (c.unsqueeze(-1) * p).clamp_min(ALPHA_FLOOR)
    t0 = tt.sum(-1); b0 = b.sum(-1)
    return (torch.lgamma(t0) - torch.lgamma(tt).sum(-1)
            - torch.lgamma(b0) + torch.lgamma(b).sum(-1)
            + ((tt - b) * (torch.digamma(tt)
                           - torch.digamma(t0).unsqueeze(-1))).sum(-1))


def dirichlet_multinomial_loss(p, c, t):
    return _value_nll(p, c, t).mean()

def dirichlet_kl_loss(p, c, t):
    return _value_kl(p, c, t).mean()

LOSS_FN = 'evidence'          # value head: 'evidence' | 'kl'

def _value_loss(p, c, t):
    return (_value_nll if LOSS_FN == 'evidence' else _value_kl)(p, c, t).mean()


def _z_ce(vp, z_col):
    """Single-draw game-outcome likelihood = cross-entropy of the value MEAN
    `vp` (B,3) against the observed outcome column `z_col` (B,), long, −1 where
    no outcome was observed (cutoffs, solver-proven, aux). This is exactly the
    Dirichlet-Multinomial NLL of one draw (P(z)=p_z), so it trains the mean
    toward the outcome WITHOUT rewarding concentration — the piece that keeps z
    from inflating α₀. Zero (graph-preserving) when the batch has no outcomes."""
    mask = z_col >= 0
    if not bool(mask.any()):
        return vp.sum() * 0.0
    idx = mask.nonzero(as_tuple=True)[0]
    p = vp.index_select(0, idx).clamp_min(1e-9)
    z = z_col.index_select(0, idx).view(-1, 1)
    return -torch.log(p.gather(1, z).squeeze(1)).mean()


def _policy_nll(ps, pc, seg, visit, B):
    """Per-sample negative Dirichlet-Multinomial log-likelihood of the visit
    counts under Dir(β·P), P = segment-softmax of the gathered legal logits
    `ps` (M,), β = `pc` (B,). `seg` (M,) maps each gathered entry to its
    sample. Fully vectorized via segment (scatter) reductions. Returns (B,)."""
    segmax = torch.full((B,), -1e30).scatter_reduce(0, seg, ps, reduce='amax',
                                                     include_self=True)
    e = torch.exp(ps - segmax[seg])
    denom = torch.zeros(B).scatter_add(0, seg, e)
    P = e / denom[seg].clamp_min(1e-30)
    A = (pc[seg] * P).clamp_min(ALPHA_FLOOR)
    n = visit
    sumA = torch.zeros(B).scatter_add(0, seg, A)
    sumn = torch.zeros(B).scatter_add(0, seg, n)
    t1 = torch.zeros(B).scatter_add(0, seg, torch.lgamma(A + n))
    t3 = torch.zeros(B).scatter_add(0, seg, torch.lgamma(A))
    log_ev = (t1 - torch.lgamma(sumA + sumn)) - (t3 - torch.lgamma(sumA))
    return -log_ev


_KL_SPLIT = (_BACKEND == 'directml')

def loss_backward(vp, vc, pl, pc, flat_idx, seg, v_counts, visit, z_col, B):
    """Combined value + z + policy loss and one backward into the shared trunk.
    `vp`(B,3)/`vc`(B,)/`pl`(B,A)/`pc`(B,) are live on the training device with
    grad; `flat_idx`(M,) int64 CPU indexes the flattened (B·A) policy axis at
    the legal entries; `seg`(M,) their sample ids; `v_counts`(B,3) and
    `visit`(M,) the targets. Returns (total, value, policy losses + detached
    CPU value predictions for diagnostics)."""
    dev = vp.device
    if _KL_SPLIT:
        # Gather policy logits to legal on-device (index_select backward = the
        # dense (B,A) policy head's only device autograd edge) or download once.
        if _GATHER_BWD_OK and str(dev) != 'cpu':
            fi = flat_idx.to(dev)
            p_sel = pl.reshape(-1).index_select(0, fi)          # (M,) device graph
            ps_cpu = p_sel.detach().cpu().requires_grad_(True)
        else:
            p_all = pl.detach().cpu().reshape(-1)
            ps_cpu = p_all.index_select(0, flat_idx).requires_grad_(True)
        vp_cpu = vp.detach().cpu().requires_grad_(True)
        vc_cpu = vc.detach().cpu().requires_grad_(True)
        pc_cpu = pc.detach().cpu().requires_grad_(True)
        vloss = _value_loss(vp_cpu, vc_cpu, v_counts)
        zloss = _z_ce(vp_cpu, z_col)
        ploss = _policy_nll(ps_cpu, pc_cpu, seg, visit, B).mean()
        total = (VALUE_LOSS_WEIGHT * vloss + Z_LOSS_WEIGHT * zloss
                 + POLICY_LOSS_WEIGHT * ploss)
        total.backward()
        outs  = [vp, vc, pc]
        grads = [vp_cpu.grad.to(dev), vc_cpu.grad.to(dev), pc_cpu.grad.to(dev)]
        if _GATHER_BWD_OK and str(dev) != 'cpu':
            outs.append(p_sel); grads.append(ps_cpu.grad.to(dev))
        else:
            gp = torch.zeros(pl.numel()).index_add_(0, flat_idx, ps_cpu.grad)
            outs.append(pl); grads.append(gp.reshape(pl.shape).to(dev))
        torch.autograd.backward(outs, grads)
        return (float(total.item()), float(vloss.item()), float(ploss.item()),
                float(zloss.item()), vp_cpu.detach(), vc_cpu.detach())
    fi   = flat_idx.to(dev)
    segd = seg.to(dev); vcnt = v_counts.to(dev); vis = visit.to(dev)
    zc   = z_col.to(dev)
    p_sel = pl.reshape(-1).index_select(0, fi)
    vloss = _value_loss(vp, vc, vcnt)
    zloss = _z_ce(vp, zc)
    ploss = _policy_nll(p_sel, pc, segd, vis, B).mean()
    total = (VALUE_LOSS_WEIGHT * vloss + Z_LOSS_WEIGHT * zloss
             + POLICY_LOSS_WEIGHT * ploss)
    total.backward()
    return (float(total.item()), float(vloss.item()), float(ploss.item()),
            float(zloss.item()), vp.detach().cpu(), vc.detach().cpu())


# Self-play source: worker pool (multiprocess + GPU server) or in-process.
import os
import threading
try:
    self_play.shutdown()
except (NameError, AttributeError):
    pass
if USE_WORKERS:
    _wcfg = dict(seed=int(np.random.randint(1_000_000)),
                 games_per_worker=GAMES_PER_WORKER, wave=WORKER_WAVE,
                 fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS,
                 fast_prob=FAST_GAME_PROB, temp_threshold=TEMP_THRESHOLD,
                 conf_cap=CONF_CAP, pool_prob=SELFPLAY_POOL_PROB,
                 checkpoint_dir=CHECKPOINT_DIR,
                 max_selfplay_plies=MAX_SELFPLAY_PLIES, restart_prob=RESTART_PROB,
                 restart_k_min=RESTART_K_MIN, restart_k_max=RESTART_K_MAX,
                 restart_pool_cap=RESTART_POOL_CAP, swing_thresh=SWING_THRESH,
                 random_pool_frac=RANDOM_POOL_FRAC)
    self_play = MPSelfPlayPool(network, DEVICE, SELFPLAY_WORKERS, _wcfg,
                               checkpoint_dir=CHECKPOINT_DIR,
                               channels=CHANNELS, num_blocks=NUM_BLOCKS)
    torch.set_num_threads(max(2, (os.cpu_count() or 6) - SELFPLAY_WORKERS))
else:
    self_play = ThompsonParallelSelfPlay(game, network, DEVICE,
                                         n_parallel=N_PARALLEL_GAMES,
                                         wave_per_game=WAVE_PER_GAME,
                                         fast_sims=FAST_MCTS_SIMS,
                                         full_sims=FULL_MCTS_SIMS,
                                         fast_prob=FAST_GAME_PROB,
                                         temp_threshold=TEMP_THRESHOLD,
                                         conf_cap=CONF_CAP,
                                         pool_prob=SELFPLAY_POOL_PROB,
                                         checkpoint_dir=CHECKPOINT_DIR,
                                         channels=CHANNELS,
                                         num_blocks=NUM_BLOCKS,
                                         max_selfplay_plies=MAX_SELFPLAY_PLIES,
                                         restart_prob=RESTART_PROB,
                                         restart_k_min=RESTART_K_MIN,
                                         restart_k_max=RESTART_K_MAX,
                                         restart_pool_cap=RESTART_POOL_CAP,
                                         swing_thresh=SWING_THRESH,
                                         random_pool_frac=RANDOM_POOL_FRAC)
episode_stream = self_play.episodes()
MODEL_LOCK = getattr(self_play, 'lock', None) or threading.Lock()

EVAL_DEVICE = 'cpu'

def _cpu_sd(net):
    return {k: v.detach().cpu() for k, v in net.state_dict().items()}

def _cpu_clone(net):
    m = ThompsonHybridChessNet(CHANNELS, NUM_BLOCKS)
    m.load_state_dict(_cpu_sd(net))
    m.eval()
    return m

def _to_cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_tree(v) for v in obj]
    return obj

# Replay buffer entries: (obs fp16, v_counts (3,), legal_idx (m,),
# visit_counts (m,), solved, on_path). `weights` is a PARALLEL array of the
# prioritized-replay sampling weight per entry, maintained in lockstep so
# minibatches can be drawn ∝ weight without an O(N) rebuild each step.
replay_buffer = []
buf_weights   = []
_init_snap = _cpu_clone(base_network)
eval_net   = _cpu_clone(base_network)
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# ONE shared Elo table keyed by player-key (checkpoint × sim-budget); 'random'
# is a single player. Keys: 'random', and f'{label}@{sims}' per benchmark×sims.
def _pkey(label, sims):
    return 'random' if label == 'random' else f'{label}@{sims}'

def _eval_players():
    """Build the current eval-player pool from `benchmarks` × EVAL_SIMS +
    random. Each carries its checkpoint net and sim budget."""
    players = [{'key': 'random', 'label': 'random', 'sims': 0, 'net': None}]
    for b in benchmarks:
        if b['label'] == 'random':
            continue
        for s in EVAL_SIMS:
            players.append({'key': _pkey(b['label'], s), 'label': b['label'],
                            'sims': s, 'net': b['net']})
    return players

elos = {'random': START_ELO}
wdl  = defaultdict(lambda: [0, 0, 0])
hist = {'ep': [], 'loss': [], 'val_loss': [], 'pol_loss': [], 'z_loss': [],
        'v_err': [], 'conf_t': [], 'conf_p': [], 'lam': [],
        'draw_pct': [], 'cut_pct': [], 'dec_pct': [], 'mean_plies': [],
        'buf_total': [], 'buf_solved': [], 'restart_pool': [],
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}
_prev_stats = {}

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_LATEST_CKPT = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def _save_benchmark_net(label, net):
    if net is not None:
        torch.save(net.state_dict(),
                   os.path.join(CHECKPOINT_DIR, f'bench_{label}.pt'))

def save_checkpoint(ep):
    with MODEL_LOCK:
        _model_sd = _to_cpu_tree(base_network.state_dict())
        _opt_sd   = _to_cpu_tree(optimizer.state_dict())
    torch.save({'ep': ep,
                'model': _model_sd,
                'optimizer': _opt_sd,
                'scheduler': scheduler.state_dict(),
                'elos': dict(elos),
                'wdl': [[ka, kb, c[0], c[1], c[2]] for (ka, kb), c in wdl.items()],
                'hist': hist,
                'bench_labels': [b['label'] for b in benchmarks]}, _LATEST_CKPT)

start_ep = 1
if RESUME_FROM_CHECKPOINT and os.path.exists(_LATEST_CKPT):
    try:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=True)
    except Exception:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=False)
    base_network.load_state_dict(_ck['model'])
    optimizer.load_state_dict(_ck['optimizer'])
    scheduler.load_state_dict(_ck['scheduler'])
    hist = _ck['hist']
    hist.setdefault('z_loss', [float('nan')] * len(hist['ep']))
    benchmarks = [{'label': 'random', 'net': None}]
    for _lbl in dict.fromkeys(_ck['bench_labels']):
        if _lbl == 'random':
            continue
        _bn = ThompsonHybridChessNet(CHANNELS, NUM_BLOCKS)
        _bn.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, f'bench_{_lbl}.pt'),
            map_location='cpu', weights_only=True))
        _bn.eval()
        benchmarks.append({'label': _lbl, 'net': _bn})
    # Eval Elo pool: load the single-table format; DISCARD an older per-track
    # checkpoint's eval ratings (format changed) — training weights are intact,
    # ratings just refill from START_ELO over the next deep evals.
    _elos_ck = _ck.get('elos', {})
    if _elos_ck and not any(isinstance(v, dict) for v in _elos_ck.values()):
        elos = dict(_elos_ck)
        wdl  = defaultdict(lambda: [0, 0, 0])
        for _row in _ck.get('wdl', []):
            if isinstance(_row, (list, tuple)) and len(_row) == 5:
                wdl[(_row[0], _row[1])] = [_row[2], _row[3], _row[4]]
    else:                                    # old per-track ratings → reset
        elos = {'random': START_ELO}
        wdl  = defaultdict(lambda: [0, 0, 0])
        hist['deep_ep'] = []; hist['elo_snap'] = []
    for _p in _eval_players():               # ensure every current key is rated
        elos.setdefault(_p['key'], START_ELO)
    start_ep = _ck['ep'] + 1
    print(f'Resumed from checkpoint: episode {_ck["ep"]}, pool {len(benchmarks)}')


def _run_eval(new_keys):
    """Rate the current pool after adding `new_keys`, then snapshot the table."""
    players = _eval_players()
    for _p in players:
        elos.setdefault(_p['key'], START_ELO)
    run_eval_pool(players, elos, wdl, game, EVAL_DEVICE, new_keys,
                  k=K_FACTOR, games_per_pair=EVAL_GAMES_PER_PAIR,
                  random_mult=EVAL_RANDOM_MULT, opening_plies=EVAL_OPENING_PLIES)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append(dict(elos))


def _print_ladder(prefix, top=16):
    players = _eval_players()
    order = sorted(players, key=lambda p: -elos[p['key']])
    ladder = '  '.join(f'{p["key"]}={elos[p["key"]]:.0f}' for p in order[:top])
    more = f'  …(+{len(order) - top} more)' if len(order) > top else ''
    print(f'{prefix} pool Elos (top {min(top, len(order))}): {ladder}{more}')
    # Current model's search benefit, if it is in the pool.
    cur = str(ep_now[0]) if ep_now[0] > 0 else '0'
    if _pkey(cur, EVAL_SIMS[0]) in elos:
        by_sim = '  '.join(f'@{s}={elos[_pkey(cur, s)]:.0f}' for s in EVAL_SIMS)
        print(f'{prefix} current model ({cur}) by sims: {by_sim}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  ThompsonHybridChessNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims (1 sim = 1 leaf open) | value+policy heads '
      f'| mixture-prop + MCTS-Solver | value {LOSS_FN}+z loss, '
      f'conf cap {CONF_CAP:.0f} | z-CE w{Z_LOSS_WEIGHT:.2f} | '
      f'pool {SELFPLAY_POOL_PROB:.0%}(rand {RANDOM_POOL_FRAC:.0%}) | '
      f'restart {RESTART_PROB:.0%} | prio pl/sv {PRIORITY_PLAYED:.1f}/{PRIORITY_SOLVER:.1f} '
      f'| {TRAIN_STEPS_PER_EP} steps/ep')
print(f'Quick eval every {EVAL_EVERY} eps; deep eval-pool every {DEEP_EVAL_EVERY} '
      f'eps | sim budgets: {EVAL_SIMS} (one shared Elo table)')
print(f'Checkpoints: {CHECKPOINT_DIR} (resume={RESUME_FROM_CHECKPOINT})')
if USE_WORKERS:
    print(f'Self-play: {SELFPLAY_WORKERS} worker processes × {GAMES_PER_WORKER} games '
          f'(wave {WORKER_WAVE}) → GPU inference server on {DEVICE}')
print('-' * 72)

aux_total = 0
ep_now = [start_ep - 1]
if start_ep == 1:
    _save_benchmark_net('0', _init_snap)
    for _p in _eval_players():
        elos.setdefault(_p['key'], START_ELO)
    _run_eval([_pkey('0', s) for s in EVAL_SIMS])
    _print_ladder('Iter     0 |')
    save_checkpoint(0)
print('-' * 72)

for ep in range(start_ep, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Pull the next finished self-play game: per-move root samples (value +
    #    policy targets) tagged played-path, plus solver-labeled aux samples.
    network.eval()
    raw_data = next(episode_stream)
    _n_aux_ep = int(getattr(self_play, 'last_aux', 0))
    aux_total += _n_aux_ep

    # 2. Extend the buffer + its parallel priority-weight array. Samples are
    #    (obs, v_counts, legal_idx, visit_counts, solved, on_path, z_col).
    for smp in raw_data:
        replay_buffer.append(smp)
        buf_weights.append(PRIORITY_SOLVER if smp[4] else PRIORITY_PLAYED)
    if len(replay_buffer) > MAX_BUFFER:
        _cut = len(replay_buffer) - MAX_BUFFER
        del replay_buffer[:_cut]
        del buf_weights[:_cut]

    # 3. Train: combined value + z + policy loss on prioritized minibatches.
    network.train()
    losses, vls, pls, zls, v_errs, conf_ts, conf_ps = [], [], [], [], [], [], []
    if len(replay_buffer) >= BATCH_SIZE:
        _w = np.asarray(buf_weights, dtype=np.float64)
        _w = _w / _w.sum()
        for _ in range(TRAIN_STEPS_PER_EP):
            idx = np.random.choice(len(replay_buffer), BATCH_SIZE,
                                   replace=False, p=_w)
            batch = [replay_buffer[k] for k in idx]
            # Value targets: dense (B,3) + the single-draw outcome column (B,).
            # Policy targets: gathered legal entries — flat (B·A) indices +
            # segment ids + visit counts.
            v_np = np.stack([b[1] for b in batch]).astype(np.float32)
            z_np = np.asarray([b[6] for b in batch], dtype=np.int64)
            fi_parts, seg_parts, vis_parts = [], [], []
            for bi, b in enumerate(batch):
                li = b[2]
                fi_parts.append(li.astype(np.int64) + bi * _NUM_ACTIONS)
                seg_parts.append(np.full(len(li), bi, dtype=np.int64))
                vis_parts.append(b[3])
            flat_idx = torch.from_numpy(np.concatenate(fi_parts))
            seg      = torch.from_numpy(np.concatenate(seg_parts))
            visit    = torch.from_numpy(np.concatenate(vis_parts).astype(np.float32))
            v_counts = torch.from_numpy(v_np)
            z_col    = torch.from_numpy(z_np)
            with MODEL_LOCK:
                x_b = batch_to_tensor([b[0] for b in batch], DEVICE)
                vp, vc, pl, pc = network(x_b)
                optimizer.zero_grad()
                tot, vl, plv, zl, vp_cpu, vc_cpu = loss_backward(
                    vp, vc, pl, pc, flat_idx, seg, v_counts, visit, z_col,
                    BATCH_SIZE)
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()
            with torch.no_grad():
                t  = v_counts.clamp_min(ALPHA_FLOOR)
                t0 = t.sum(dim=-1)
                tv = (t[:, 0] - t[:, 2]) / t0
                pv = vp_cpu[:, 0] - vp_cpu[:, 2]
                v_errs.append(float((tv - pv).abs().mean()))
                conf_ts.append(float(t0.mean()))
                conf_ps.append(float(vc_cpu.mean()))
            losses.append(tot); vls.append(vl); pls.append(plv); zls.append(zl)
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation (CPU replica).
    network.eval()
    with MODEL_LOCK:
        eval_net.load_state_dict(_cpu_sd(base_network))
    mloss = float(np.mean(losses)) if losses else float('nan')
    mvl   = float(np.mean(vls))    if vls    else float('nan')
    mpl   = float(np.mean(pls))    if pls    else float('nan')
    mzl   = float(np.mean(zls))    if zls    else float('nan')
    mve   = float(np.mean(v_errs)) if v_errs else float('nan')
    mct   = float(np.mean(conf_ts)) if conf_ts else float('nan')
    mcp   = float(np.mean(conf_ps)) if conf_ps else float('nan')
    lam_now = float(getattr(self_play, 'last_lam', float('nan')))
    _st  = dict(getattr(self_play, 'stats', {}))
    _dg  = _st.get('games', 0)   - _prev_stats.get('games', 0)
    _dd  = _st.get('draw', 0)    - _prev_stats.get('draw', 0)
    _dc  = _st.get('cutoff', 0)  - _prev_stats.get('cutoff', 0)
    _dp  = _st.get('plies', 0)   - _prev_stats.get('plies', 0)
    _prev_stats = _st
    draw_pct = 100.0 * _dd / _dg if _dg else float('nan')
    cut_pct  = 100.0 * _dc / _dg if _dg else float('nan')
    dec_pct  = 100.0 * (_dg - _dd - _dc) / _dg if _dg else float('nan')
    mean_plies = _dp / _dg if _dg else float('nan')
    buf_total  = len(replay_buffer)
    buf_solved = sum(1 for _s in replay_buffer if _s[4])
    restart_pool = int(getattr(self_play, 'last_restart_pool', 0))
    hist['ep'].append(ep); hist['loss'].append(mloss)
    hist['val_loss'].append(mvl); hist['pol_loss'].append(mpl)
    hist['z_loss'].append(mzl)
    hist['v_err'].append(mve)
    hist['conf_t'].append(mct); hist['conf_p'].append(mcp)
    hist['lam'].append(lam_now)
    hist['draw_pct'].append(draw_pct); hist['cut_pct'].append(cut_pct)
    hist['dec_pct'].append(dec_pct);   hist['mean_plies'].append(mean_plies)
    hist['buf_total'].append(buf_total); hist['buf_solved'].append(buf_solved)
    hist['restart_pool'].append(restart_pool)
    _slv_pct = 100.0 * buf_solved / buf_total if buf_total else 0.0
    _diag = (f'dr={draw_pct:.0f}% cut={cut_pct:.0f}% ply={mean_plies:.0f} '
             f'buf={buf_total/1000:.0f}k/{_slv_pct:.0f}%slv rp={restart_pool}')
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        new_label = str(ep)
        with MODEL_LOCK:
            snap = _cpu_clone(base_network)
        _existing = next((b for b in benchmarks if b['label'] == new_label), None)
        if _existing is not None:
            _existing['net'] = snap
        else:
            benchmarks.append({'label': new_label, 'net': snap})
        # Warm-start each new player's Elo from the previous generation's same-
        # sim rating (falls back to START_ELO), then rate the whole pool.
        _prev = benchmarks[-2]['label'] if len(benchmarks) >= 2 else 'random'
        _new_keys = []
        for s in EVAL_SIMS:
            nk = _pkey(new_label, s)
            elos[nk] = elos.get(nk, elos.get(_pkey(_prev, s), START_ELO))
            _new_keys.append(nk)
        _run_eval(_new_keys)
        print(f'Ep {ep:5d} | L={mloss:.3f}(v{mvl:.3f}/z{mzl:.3f}/p{mpl:.3f}) vErr={mve:.3f} '
              f'a0 t/p={mct:.1f}/{mcp:.1f} lam={lam_now:.2f} aux={aux_total} {_diag} '
              f'| DEEP eval-pool ({len(_eval_players())} players) | lr={lr_now:.2e}')
        _print_ladder('        |')
        _save_benchmark_net(new_label, snap)
        save_checkpoint(ep)
    else:
        ref = benchmarks[-1]
        w, d, l = play_match(eval_net, ref['net'], game,
                             QUICK_EVAL_GAMES, EVAL_DEVICE)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        _ref0 = elos.get(_pkey(ref['label'], 0), float('nan'))
        print(f'Ep {ep:5d} | L={mloss:.3f}(v{mvl:.3f}/z{mzl:.3f}/p{mpl:.3f}) vErr={mve:.3f} '
              f'a0 t/p={mct:.1f}/{mcp:.1f} lam={lam_now:.2f} aux={aux_total} {_diag} '
              f'| vs {ref["label"]:>5} W{w:2d}D{d:2d}L{l:2d} '
              f'(@0 Elo={_ref0:.0f}) | lr={lr_now:.2e}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
fig.suptitle('Chess ThompsonZero Hybrid (state-value + policy prior) Training',
             fontsize=14)

# Value + policy losses, with the value α₀ target/predicted on a twin axis.
ax = axes[0, 0]
ax.plot(hist['ep'], hist['val_loss'], color='tab:blue', label='value loss')
ax.plot(hist['ep'], hist['pol_loss'], color='tab:red', label='policy loss')
if hist.get('z_loss'):
    ax.plot(hist['ep'], hist['z_loss'], color='tab:green', label='z CE')
ax.set_title(f'Value ({LOSS_FN}) + z-CE + policy losses')
ax.set_xlabel('Episode')
ax2 = ax.twinx()
ax2.plot(hist['ep'], hist['conf_t'], color='tab:purple', alpha=0.4, label='α₀ target')
ax2.plot(hist['ep'], hist['conf_p'], color='tab:brown', alpha=0.4, label='α₀ pred')
ax2.set_ylabel('mean value α₀')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8)

axes[0, 1].plot(hist['ep'], hist['v_err'], color='tab:orange', label='|v err|')
axes[0, 1].set_title('|value target − predicted|  +  calibration λ')
axes[0, 1].set_xlabel('Episode')
if any(np.isfinite(hist.get('lam', []))):
    axl = axes[0, 1].twinx()
    axl.plot(hist['ep'], hist['lam'], color='tab:blue', alpha=0.6, label='λ (calib)')
    axl.axhline(1.0, color='tab:blue', linestyle=':', linewidth=0.8, alpha=0.5)
    axl.set_ylabel('λ  (1 = calibrated)')
    h1, l1 = axes[0, 1].get_legend_handles_labels()
    h2, l2 = axl.get_legend_handles_labels()
    axes[0, 1].legend(h1 + h2, l1 + l2, fontsize=8)

# Current-model Elo by SIM BUDGET (one line per budget, from the shared pool).
def _cur_label(ep):
    return '0' if ep == 0 else str(ep)
for s in EVAL_SIMS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        key = _pkey(_cur_label(ep), s)
        if key in snap:
            xs.append(ep); ys.append(snap[key])
    axes[0, 2].plot(xs, ys, marker='.', label=f'@{s}')
axes[0, 2].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Current-model Elo by sim budget')
axes[0, 2].set_xlabel('Episode'); axes[0, 2].legend(fontsize=8)

axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, policy-head)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per checkpoint, grouped bars by sim budget (from the shared pool).
labels = [b['label'] for b in benchmarks if b['label'] != 'random']
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_SIMS), 1)
for k, s in enumerate(EVAL_SIMS):
    vals = [elos.get(_pkey(l, s), np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_SIMS) - 1) / 2) * width, vals, width, label=f'@{s}')
axes[1, 1].axhline(elos.get('random', START_ELO), color='gray', linestyle='--',
                   linewidth=0.8, label='random')
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per checkpoint (by sims)'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Search benefit: current-model Elo(@sims) − Elo(@0) over training. The whole
# point of the hybrid — is search adding strength, and is that gap growing?
for s in EVAL_SIMS[1:]:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        k0, ks = _pkey(_cur_label(ep), 0), _pkey(_cur_label(ep), s)
        if k0 in snap and ks in snap:
            xs.append(ep); ys.append(snap[ks] - snap[k0])
    axes[1, 2].plot(xs, ys, marker='.', label=f'@{s} − @0')
axes[1, 2].axhline(0.0, color='gray', linestyle='--', linewidth=0.8)
axes[1, 2].set_title('Search benefit (Elo over the policy head)')
axes[1, 2].set_xlabel('Episode'); axes[1, 2].set_ylabel('Elo gain from search')
axes[1, 2].legend(fontsize=8)

ax = axes[0, 3]
if hist.get('draw_pct'):
    ax.plot(hist['ep'], hist['dec_pct'], color='tab:green', label='decisive %')
    ax.plot(hist['ep'], hist['draw_pct'], color='gray', label='draw %')
    ax.plot(hist['ep'], hist['cut_pct'], color='tab:red', linestyle=':', label='cutoff %')
    ax.set_ylim(0, 100); ax.set_ylabel('% of self-play games')
    axp = ax.twinx()
    axp.plot(hist['ep'], hist['mean_plies'], color='tab:blue', alpha=0.4,
             label='mean plies')
    axp.set_ylabel('mean plies/game')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = axp.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc='center right')
ax.set_title('Self-play outcome mix'); ax.set_xlabel('Episode')

ax = axes[1, 3]
if hist.get('buf_total'):
    bt = np.array(hist['buf_total'], dtype=float)
    bs = np.array(hist['buf_solved'], dtype=float)
    ax.plot(hist['ep'], bt / 1000.0, color='tab:purple', label='buffer (k)')
    ax.plot(hist['ep'], bs / 1000.0, color='tab:orange', label='solver-labeled (k)')
    ax.set_ylabel('positions (thousands)')
    axr = ax.twinx()
    axr.plot(hist['ep'], hist['restart_pool'], color='tab:brown', alpha=0.4,
             label='restart pool')
    axr.set_ylabel('restart-pool seeds (last worker)')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = axr.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc='center right')
ax.set_title('Buffer composition + restart pool'); ax.set_xlabel('Episode')

plt.tight_layout()
plt.show()

## Arena: pit any two checkpoints against each other

There's no existing "KataZero-chess" notebook to compare against the way the
Boop notebooks compare ThompsonZero to KataZero, so this section is a general
**checkpoint-vs-checkpoint arena** instead: load any two `ThompsonChessNet`
state dicts (any `bench_<ep>.pt` from `CHECKPOINT_DIR`, or `latest.pt`'s
`['model']` entry) and play them off, search-free or with search, alternating
colours, with a printable move-by-move game log (chess positions are
human-inspectable via FEN/SAN, unlike Boop's board).

The training cell's own DEEP-eval tournaments already give you an ongoing Elo
ladder across generations (that *is* the primary strength signal — see the
plots cell); this section is for **ad-hoc** comparisons: two specific
checkpoints, a specific `LOSS_FN` variant against another, or a specific sim
budget.

**Plugging in an external opponent** (Stockfish via UCI, another engine, a
human): `arena()` below is specifically for two `ThompsonChessNet` checkpoints.
To face something else, write your own move loop around a `pick(state) ->
action` function for the external side (any `state.legal_actions()`-compatible
choice) — `arena()`'s body is short and is a template for that; nothing here
has been tested against an external engine, so treat this as a starting point,
not a ready integration.

In [ ]:
import os


def _load_sd(path):
    try:
        sd = torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        sd = torch.load(path, map_location='cpu', weights_only=False)
    return sd['model'] if isinstance(sd, dict) and 'model' in sd else sd


def load_chess_net(path, channels=CHANNELS, num_blocks=NUM_BLOCKS):
    net = ThompsonHybridChessNet(channels, num_blocks)
    net.load_state_dict(_load_sd(path))
    net.eval()
    return net


def arena(net_a, net_b, game, n_games=20, a_sims=0, b_sims=0,
          device='cpu', opening_plies=4, verbose=True):
    """net_a vs net_b. sims == 0 -> search-free policy-head argmax; sims > 0 ->
    full hybrid Thompson-MCTS (posterior-mean argmax over opened edges).
    Alternates colours; the first `opening_plies` moves are random for variety.
    Returns (wins_a, draws, wins_b) from net_a's perspective."""
    bot_a = (make_thompson_bot(game, net_a, device, a_sims,
                               batch_size=max(1, min(a_sims, 16)))
             if a_sims > 0 else None)
    bot_b = (make_thompson_bot(game, net_b, device, b_sims,
                               batch_size=max(1, min(b_sims, 16)))
             if b_sims > 0 else None)
    w = d = l = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            if ply < opening_plies:
                action = random.choice(state.legal_actions())
            elif state.current_player() == a_player:
                action = (_mcts_move(bot_a, state) if bot_a is not None
                          else policy_action(net_a, state, device))
            else:
                action = (_mcts_move(bot_b, state) if bot_b is not None
                          else policy_action(net_b, state, device))
            state.apply_action(action)
            ply += 1
        r = state.returns()[a_player]
        if r > 0:   w += 1
        elif r < 0: l += 1
        else:       d += 1
        if verbose and (i + 1) % 5 == 0:
            print(f'    {i + 1:3d}/{n_games}  net_a W{w} D{d} L{l}')
    return w, d, l


def play_and_show(net_a, net_b, game, device='cpu', max_plies=200):
    """One game, net_a (White) vs net_b (Black), search-free, printing SAN and
    the final result. Either net may be None for a random mover."""
    state = game.new_initial_state()
    moves = []
    ply = 0
    while not state.is_terminal() and ply < max_plies:
        mover = state.current_player()
        net = net_a if mover == 1 else net_b     # player 1 = White (see cell 2)
        action = (random.choice(state.legal_actions()) if net is None
                  else policy_action(net, state, device))
        moves.append(state.action_to_string(mover, action))
        state.apply_action(action)
        ply += 1
    print(' '.join(f'{i//2+1}.{m}' if i % 2 == 0 else m
                   for i, m in enumerate(moves)))
    if state.is_terminal():
        print('Result:', state.returns(), '  FEN:', state)
    else:
        print(f'(stopped at the {max_plies}-ply display cap, not terminal)')
    return state


# ── Run it ────────────────────────────────────────────────────────────────────
CKPT_A       = os.path.join(CHECKPOINT_DIR, 'latest.pt')   # newest
CKPT_B       = os.path.join(CHECKPOINT_DIR, 'bench_0.pt')  # untrained baseline
ARENA_GAMES  = 20
ARENA_SIMS   = 50     # per-move simulations for the with-search match
ARENA_DEVICE = 'cpu'

if not os.path.exists(CKPT_A):
    print(f'No checkpoint at {CKPT_A} yet — train at least one episode first.')
else:
    net_a = load_chess_net(CKPT_A)
    net_b = load_chess_net(CKPT_B) if os.path.exists(CKPT_B) else None
    print(f'net_a: {CKPT_A}')
    print(f'net_b: {CKPT_B if net_b is not None else "(missing — using random)"}')

    if net_b is not None:
        print(f'\nSearch-free policy-head ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES, 0, 0, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

        print(f'\n{ARENA_SIMS}-sim hybrid MCTS ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES,
                        ARENA_SIMS, ARENA_SIMS, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

    print('\nOne search-free game vs random, shown move-by-move:')
    play_and_show(net_a, None if net_b is None else net_b, game, ARENA_DEVICE)